# Analysis

**Hypothesis**: Within the developing human heart MERFISH dataset, specific cell populations (Populations in adata.obs['Populations']) show systematic shifts in transcriptional complexity and purity-associated gene expression across samples (atria/ventricle–like regions), reflecting region-specific maturation states that are not captured by clustering alone.

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Set up visualization defaults for better plots
sc.settings.verbosity = 3
sc.settings.figsize = (8, 8)
sc.settings.dpi = 100
sc.settings.facecolor = 'white'
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 8)
plt.rcParams['savefig.dpi'] = 150
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.2)

# Load data
print("Loading data...")
adata = sc.read_h5ad("/home/mingqiam/TissueAgent/demo/data/dataset_farah_anon.h5ad")
print(f"Data loaded: {adata.shape[0]} cells and {adata.shape[1]} genes")


Loading data...


Data loaded: 228635 cells and 238 genes


# Analysis Plan

**Hypothesis**: Within the developing human heart MERFISH dataset, specific cell populations (Populations in adata.obs['Populations']) show systematic shifts in transcriptional complexity and purity-associated gene expression across samples (atria/ventricle–like regions), reflecting region-specific maturation states that are not captured by clustering alone.

## Steps:
- Inspect and summarize the distributions of Complexity, Purity, UMI Count, Populations, Batch, and Sample_ID, including stratified summaries of Complexity/Purity/UMI Count by Populations, Sample_ID, and Batch, to identify major cell populations and global differences in these quantitative metrics across samples.
- Within each major Population, formally quantify how Complexity and Purity vary across Sample_ID and Batch using per-population tests (e.g., ANOVA or Kruskal–Wallis, with effect sizes and p-values) where the response variables are Complexity and Purity, not gene expression, to detect region-associated shifts in these QC-like metrics.
- For each Population that shows significant Complexity or Purity shifts across Sample_ID, perform within-population differential expression between highest- and lowest-Complexity tertiles while stratifying by Sample_ID or otherwise controlling for Sample_ID (e.g., per-sample DE combined meta-analytically), and summarize top positive/negative genes per population in tabular form.
- For each Population with robust DE results, derive population-specific gene sets associated with higher vs lower Complexity/Purity (e.g., top positive markers from step 3) and compute per-cell scores using sc.tl.score_genes, ensuring signatures are tagged by their source Population; then test whether these scores differ systematically across Sample_ID within each relevant Population using appropriate non-parametric tests.
- Within each key Population and Sample_ID, relate inferred maturation-associated scores to spatial position by computing spatial autocorrelation statistics (e.g., Moran’s I using a k-nearest-neighbors graph built from adata.obsm['spatial'] via scipy), explicitly specifying k and restricting computations to within-Population, within-sample subsets to avoid mixing spatial scales.
- Summarize for each key Population which samples and spatial neighborhoods are enriched for high-maturation states, based on Complexity, Purity, gene expression signatures, and spatial autocorrelation, reporting these findings in text tables and concise printed descriptions without figures.


## Extend the initial descriptive step to include global and stratified summaries of Complexity, Purity, and UMI Count by Populations, Sample_ID, and Batch, along with missing-value counts and normalized population compositions, to fully characterize global and sample-level differences without performing formal hypothesis tests yet.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc

# 1. Basic overview of key metadata fields
print('adata shape:', adata.shape)
print('\n.obs columns:', list(adata.obs.columns))
print('\nFirst 5 rows of .obs:')
print(adata.obs.head())

# 2. Identify available quantitative covariates and summarize globally
quant_cols = []
for col in ['UMI Count', 'Complexity', 'Purity']:
    if col in adata.obs.columns:
        quant_cols.append(col)

if len(quant_cols) == 0:
    print('\nNo expected quantitative covariates found in adata.obs.')
else:
    print('\nGlobal summary statistics for quantitative covariates:')
    summary_df = adata.obs[quant_cols].describe()
    print(summary_df)

    # Explicitly report missing values per quantitative covariate
    print('\nNumber of missing values per quantitative covariate:')
    na_counts = adata.obs[quant_cols].isna().sum()
    print(na_counts)

# 3. Categorical covariates: Populations, Sample_ID, Batch, leiden
cat_cols = []
for col in ['Populations', 'Sample_ID', 'Batch', 'leiden']:
    if col in adata.obs.columns:
        cat_cols.append(col)

for col in cat_cols:
    print(f"\nValue counts for {col} (top 20):")
    vc = adata.obs[col].value_counts().head(20)
    print(vc)
    print(f"Total unique {col} levels:", adata.obs[col].nunique())

# 4. Cross-tabulation of Populations by Sample_ID to see population composition per sample
if 'Populations' in adata.obs.columns and 'Sample_ID' in adata.obs.columns:
    print('\nContingency table: Populations x Sample_ID (top 20 populations by size):')
    pop_counts = adata.obs['Populations'].value_counts()
    top_pops = pop_counts.head(20).index
    mask_top = adata.obs['Populations'].isin(top_pops)
    ctab = pd.crosstab(
        adata.obs.loc[mask_top, 'Populations'],
        adata.obs.loc[mask_top, 'Sample_ID']
    )
    print(ctab)

    # Column-normalized frequencies (per Sample_ID) to highlight compositional shifts
    print('\nColumn-normalized (per Sample_ID) frequencies for top populations (rounded to 3 decimals):')
    ctab_col_norm = ctab.div(ctab.sum(axis=0), axis=1)
    print(ctab_col_norm.round(3))

# 5. Correlations among quantitative covariates (global)
if len(quant_cols) >= 2:
    print('\nPairwise Pearson correlations among quantitative covariates:')
    corr = adata.obs[quant_cols].corr(method='pearson')
    print(corr)

# 6. Stratified summaries of quantitative covariates by Sample_ID, Populations, and Batch
if len(quant_cols) > 0 and 'Sample_ID' in adata.obs.columns:
    print('\nPer-Sample_ID summary of quantitative covariates (count, mean, std, median):')
    by_sample = adata.obs.groupby('Sample_ID')[quant_cols].agg(['count', 'mean', 'std', 'median'])
    print(by_sample)

if len(quant_cols) > 0 and 'Populations' in adata.obs.columns:
    print('\nPer-Population summary of quantitative covariates for top 20 populations (count, mean, std, median):')
    top_pops = adata.obs['Populations'].value_counts().head(20).index
    by_pop = (
        adata.obs.loc[adata.obs['Populations'].isin(top_pops)]
        .groupby('Populations')[quant_cols]
        .agg(['count', 'mean', 'std', 'median'])
    )
    print(by_pop)

if len(quant_cols) > 0 and 'Batch' in adata.obs.columns:
    print('\nPer-Batch summary of quantitative covariates (count, mean, std, median):')
    by_batch = adata.obs.groupby('Batch')[quant_cols].agg(['count', 'mean', 'std', 'median'])
    print(by_batch)

adata shape: (228635, 238)

.obs columns: ['Sample_ID', 'Batch', 'UMI Count', 'leiden', 'Complexity', 'Populations', 'Purity']

First 5 rows of .obs:
           Sample_ID    Batch  UMI Count  leiden  Complexity Populations  \
cell_id                                                                    
6-R77_4C4    R77_4C4  R77_4C4       86.0       6           8          PG   
8-R77_4C4    R77_4C4  R77_4C4      148.0       6           8          PG   
9-R77_4C4    R77_4C4  R77_4C4      100.0       6           8          PG   
10-R77_4C4   R77_4C4  R77_4C4       70.0       6           9          PG   
12-R77_4C4   R77_4C4  R77_4C4       63.0       6           8          PG   

              Purity  
cell_id               
6-R77_4C4   0.544534  
8-R77_4C4   0.625984  
9-R77_4C4   0.583665  
10-R77_4C4  0.766393  
12-R77_4C4  0.596838  

Global summary statistics for quantitative covariates:
           UMI Count    Complexity         Purity
count  228635.000000  228635.00000  228635.000000


         UMI Count                                Complexity             \
             count        mean         std median      count       mean   
Batch                                                                     
R77_4C4      72962  371.292878  235.195298  326.0      72962  10.242003   
R78_4C12     75782  424.835700  258.884537  380.0      75782  10.036922   
R78_4C15     79891  525.202038  324.020204  474.0      79891   9.407756   

                          Purity                                
               std median  count      mean       std    median  
Batch                                                           
R77_4C4   2.936185   10.0  72962  0.497966  0.153822  0.483553  
R78_4C12  2.846728   10.0  75782  0.499514  0.154452  0.491003  
R78_4C15  2.827841   10.0  79891  0.509529  0.148292  0.496032  


### Agent Interpretation

The current summaries already support the core idea that “QC-like” metrics (Complexity, Purity, UMI Count) are not homogeneous and are structured both by population and by sample, so the hypothesis is plausible and worth pursuing with formal tests.

Key points and implications:

1. **QC metrics are informative and non-redundant**
   - Complexity and Purity are strongly *negatively* correlated (r ≈ –0.57), and Complexity vs UMI Count is mildly negative (r ≈ –0.21) with almost no correlation between Purity and UMI Count.
   - This means Complexity and Purity capture different axes than simple depth, so using them as “maturation-like” axes is justified and not just a proxy for UMI Count.

2. **Strong population-level heterogeneity in Complexity and Purity**
   - Per-population means span wide ranges:
     - Complexity mean from ~5.8 (PS) / ~6.2 (PB) up to ~12.9 (PK) and ~12.6 (PN).
     - Purity mean from ~0.39 (PJ), ~0.41 (PL), ~0.42 (PD/PH) up to ~0.70 (PB), ~0.66 (PS), ~0.64 (PG), ~0.62 (PR), etc.
   - Many populations show high Purity but low Complexity (e.g., PB, PS, PG), which may represent more “mature/specialized” profiles in your working model; others have high Complexity but lower Purity (PK, PN, PO, PJ), more “transcriptionally diverse/immature.”
   - This heterogeneity itself is consistent with maturation-state differences across populations, independent of clustering.

3. **Clear compositional shifts of populations across samples**
   - The Populations × Sample_ID table shows substantial compositional differences. Examples:
     - PI fraction: 0.074 in R77_4C4 vs 0.025 in R78_4C15.
     - PL: 0.032 in R77_4C4 vs 0.056 in R78_4C15.
     - PG: 0.066 in R77 vs 0.045–0.050 in the R78 samples.
   - This confirms region/sample-specific enrichment/depletion of particular populations, which is important because later you’ll want to avoid confounding “maturation shift” with “we just have different populations.”

4. **Sample-level shifts in QC metrics**
   - UMI Count means: 371 (R77_4C4) < 425 (R78_4C12) < 525 (R78_4C15); medians similarly shift.
   - Complexity means: 10.24 (R77_4C4), 10.04 (R78_4C12), 9.41 (R78_4C15) – a monotonic *decrease*.
   - Purity means: 0.498, 0.500, 0.510 – a modest but consistent *increase*.
   - This pattern (more reads, lower Complexity, higher Purity in R78_4C15) fits nicely with a potential maturation axis at the sample level (e.g., a more specialized or regionally distinct sample), but it’s also potentially confounded with technical batch.

Overall, these patterns are promising for your hypothesis: QC metrics are structured by both population and sample in a non-trivial way, and not simply tracking depth.

Suggestions for next steps (to sharpen and deconvolve biological vs technical effects):

1. **Move to step 2: per-population tests across Sample_ID/Batch**
   - For each reasonably large population (e.g., the top ~15–20 by size you’ve already identified), fit simple models:
     - For Complexity: `Complexity ~ Sample_ID` (or non-parametric Kruskal–Wallis across the 3 samples).
     - For Purity: similarly, `Purity ~ Sample_ID`.
   - Because Batch = Sample_ID here, you can’t separate batch from region, but that’s acceptable for the current hypothesis (sample = anatomical region, albeit confounded with batch).
   - Output:
     - Test statistic, p-value, effect size (e.g., eta² or partial eta²).
     - Per-sample means and medians per population.
   - Focus attention on populations where:
     - The across-sample effect is statistically strong (e.g., FDR < 0.05).
     - The effect size is non-trivial and the direction is consistent with a maturation interpretation (e.g., clear monotone trend in Complexity or Purity across samples).

2. **Identify “candidate maturation-populations”**
   - Based on step 2, pick:
     - Populations with clear *within-population* shifts in Complexity across samples (e.g., Complexity R77 > R78_4C12 > R78_4C15, or the reverse).
     - Populations with similar patterns in Purity (e.g., Purity gradually increasing or decreasing with sample).
   - Cross-check that these populations have sufficient cells in each sample to support later DE (e.g., ≥200–300 cells per sample and ≥100 cells in each Complexity tertile).

3. **Be cautious with extreme populations**
   - Some populations already have very low mean Complexity (PB ≈ 6.2, PS ≈ 5.8) and high Purity; others have very high Complexity and moderate Purity (PK, PN, PO).
   - For these, before interpreting as maturation:
     - Check that the within-population sample-wise distributions aren’t completely overlapping (i.e., that there are real shifts, not just overall differences between populations).
   - Also, the global negative correlation between Complexity and Purity implies that shifts in one often come with opposite shifts in the other; in step 2, consider analyzing them jointly (e.g., bivariate summaries or MANOVA-style tests) for the star populations.

4. **Prepare for step 3 (DE by Complexity tertiles)**
   - Once you have populations with robust Complexity/Purity shifts across samples:
     - Within each such population, precompute per-sample Complexity tertiles and ensure that “high” and “low” groups exist in *each* sample (to allow per-sample DE and meta-analysis).
     - Pay attention to populations where Complexity distributions are highly compressed or bimodal—those may yield unstable tertiles.
   - Since UMI Count is modestly anti-correlated with Complexity, you should plan to:
     - Either include UMI Count as a covariate in the DE models (if using something like `scanpy.tl.rank_genes_groups` you may approximate this by pre-regressing UMI Count from log-normalized counts).
     - Or at least check that UMI Count distributions do not differ drastically between high- and low-Complexity cells within each sample.

5. **Keep the analysis distinct from the original paper**
   - You’re focusing explicitly on Complexity/Purity as “latent maturation/QC axes” and then deriving gene signatures and spatial autocorrelation from them. This is already conceptually distinct from standard clustering or spatial neighborhood analyses that the paper might have done.
   - Continue to emphasize:
     - Within-population, across-sample shifts in these QC-like metrics.
     - Gene signatures tied explicitly to these shifts.
     - Spatial autocorrelation of maturity scores *within population and sample*.

In summary, the initial step strongly supports continuing: QC metrics differ meaningfully by population and sample, and they’re not trivially explained by UMI Count. The next critical move is the formal, per-population statistical testing across Sample_ID to identify a focused set of populations where Complexity/Purity truly encode region-associated maturation differences, which will then drive the downstream DE and spatial analyses.

## Next Steps
Step 1: Within each sufficiently large Population, formally test whether Complexity and Purity differ across Sample_ID (and thus Batch) using non-parametric Kruskal–Wallis tests, compute simple effect sizes, and summarize per-sample means/medians to identify populations with the strongest within-population shifts.
Step 2: From these results, select a focused subset of Populations that show robust, interpretable across-sample shifts in Complexity and/or Purity (e.g., FDR < 0.05, non-trivial effect sizes, and monotone or otherwise structured trends), explicitly recording the joint directions of Complexity and Purity changes as candidate maturation populations.
Step 3: Within each selected candidate Population, define per-sample Complexity tertiles, perform within-sample differential expression between highest and lowest Complexity tertiles, and meta-analyze gene-level statistics across samples to obtain robust signatures of high- vs low-Complexity states while tracking Purity behavior in parallel.
Step 4: From the DE results, derive population-specific maturation gene signatures (e.g., top genes up in high- vs low-Complexity), score each cell with these signatures using sc.tl.score_genes, and test whether signature scores jointly with Complexity and Purity show consistent shifts across Sample_ID within each Population.
Step 5: For each key Population and Sample_ID, quantify spatial structuring of maturation scores (and Complexity/Purity where relevant) by computing Moran’s I on a spatial k-nearest-neighbor graph (per population, per sample), and summarize which regions and neighborhoods are enriched for high-maturation states, acknowledging that Sample_ID and Batch are confounded and cannot be disentangled technically vs anatomically.

## This code implements step 1 by performing within-population Kruskal–Wallis tests of Complexity and Purity across Sample_ID, computing simple eta-squared-like effect sizes, applying BH FDR correction, and printing ranked per-population results along with per-sample summaries, while incorporating robustness and clarity improvements from the feedback.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Ensure required columns exist
required_cols = ['Populations', 'Sample_ID', 'Complexity', 'Purity']
missing = [c for c in required_cols if c not in adata.obs.columns]
if len(missing) > 0:
    raise ValueError(f"Missing required columns in adata.obs: {missing}")

obs = adata.obs.copy()

# Focus on reasonably large populations to ensure stable tests
pop_counts = obs['Populations'].value_counts()
min_cells_per_pop = 300  # adjustable threshold; chosen to ensure stable non-parametric tests
large_pops = pop_counts[pop_counts >= min_cells_per_pop].index.tolist()

print(f"Number of Populations with \\u2265 {min_cells_per_pop} cells: {len(large_pops)}")
print("Large Populations:", large_pops)

results = []
skipped_insufficient_samples = 0

# Helper to compute a simple effect size (eta-squared-like) from Kruskal-Wallis
# Note: this is a heuristic, not a standard KW effect size, and is used only for ranking.
def kw_eta_squared(h_stat, total_n):
    if total_n <= 1:
        return np.nan
    return h_stat / (total_n - 1)

for pop in large_pops:
    pop_mask = obs['Populations'] == pop
    df_pop = obs.loc[pop_mask, ['Sample_ID', 'Complexity', 'Purity']].dropna()

    # Require at least 2 samples with sufficient cells to perform a test
    sample_counts = df_pop['Sample_ID'].value_counts()
    valid_samples = sorted(sample_counts[sample_counts >= 30].index.tolist())
    if len(valid_samples) < 2:
        skipped_insufficient_samples += 1
        continue

    df_pop = df_pop[df_pop['Sample_ID'].isin(valid_samples)]

    # Prepare group-wise arrays
    groups_complexity = [df_pop.loc[df_pop['Sample_ID'] == s, 'Complexity'].values for s in valid_samples]
    groups_purity = [df_pop.loc[df_pop['Sample_ID'] == s, 'Purity'].values for s in valid_samples]

    # Only test if all groups have more than 1 observation
    if any(len(g) < 2 for g in groups_complexity) or any(len(g) < 2 for g in groups_purity):
        skipped_insufficient_samples += 1
        continue

    # Kruskal-Wallis tests across Sample_ID for Complexity and Purity
    h_c, p_c = stats.kruskal(*groups_complexity)
    h_p, p_p = stats.kruskal(*groups_purity)

    # Effect size approximations
    eta2_c = kw_eta_squared(h_c, df_pop.shape[0])
    eta2_p = kw_eta_squared(h_p, df_pop.shape[0])

    # Per-sample summaries to aid interpretation
    summary = df_pop.groupby('Sample_ID').agg(
        n_cells=('Complexity', 'size'),
        complexity_mean=('Complexity', 'mean'),
        complexity_median=('Complexity', 'median'),
        purity_mean=('Purity', 'mean'),
        purity_median=('Purity', 'median')
    )

    results.append({
        'Population': pop,
        'n_total_cells': df_pop.shape[0],
        'n_samples_tested': len(valid_samples),
        'Complexity_H': h_c,
        'Complexity_pval': p_c,
        'Complexity_eta2_approx': eta2_c,
        'Purity_H': h_p,
        'Purity_pval': p_p,
        'Purity_eta2_approx': eta2_p,
        'per_sample_summary': summary
    })

# Convert to a flat DataFrame for multiple-testing correction and ranking
if len(results) == 0:
    print("No populations met size and sample-count criteria for testing.")
else:
    flat_rows = []
    for res in results:
        flat = {k: v for k, v in res.items() if k != 'per_sample_summary'}
        flat_rows.append(flat)
    res_df = pd.DataFrame(flat_rows)

    # Defensive check for NaN p-values
    if np.isnan(res_df['Complexity_pval']).any() or np.isnan(res_df['Purity_pval']).any():
        raise ValueError("NaN p-values found; inspect upstream filtering before proceeding with FDR correction.")

    # Benjamini-Hochberg FDR correction for Complexity and Purity separately
    def bh_fdr(pvals):
        pvals = np.asarray(pvals, dtype=float)
        n = pvals.size
        order = np.argsort(pvals)
        ranked_p = pvals[order]
        fdr = np.empty(n, dtype=float)
        prev = 1.0
        for i in range(n - 1, -1, -1):
            rank = i + 1
            val = min(prev, ranked_p[i] * n / rank)
            fdr[i] = val
            prev = val
        out = np.empty(n, dtype=float)
        out[order] = fdr
        return out

    res_df['Complexity_fdr'] = bh_fdr(res_df['Complexity_pval'].values)
    res_df['Purity_fdr'] = bh_fdr(res_df['Purity_pval'].values)

    # Rank populations by smallest FDR across the two metrics
    res_df['min_fdr'] = res_df[['Complexity_fdr', 'Purity_fdr']].min(axis=1)
    res_df_sorted = res_df.sort_values('min_fdr').reset_index(drop=True)

    print("\nPer-Population Kruskal-Wallis results across Sample_ID (sorted by min FDR):")
    cols_to_show = [
        'Population', 'n_total_cells', 'n_samples_tested',
        'Complexity_H', 'Complexity_pval', 'Complexity_fdr', 'Complexity_eta2_approx',
        'Purity_H', 'Purity_pval', 'Purity_fdr', 'Purity_eta2_approx',
        'min_fdr'
    ]
    print(res_df_sorted[cols_to_show].to_string(index=False, float_format=lambda x: f"{x:.3e}"))

    # Also print per-sample summaries for the top few significant populations
    top_k = min(5, res_df_sorted.shape[0])
    print(f"\nPer-sample Complexity and Purity summaries for top {top_k} Populations by min FDR:\n")
    for i in range(top_k):
        pop = res_df_sorted.loc[i, 'Population']
        per_sample = [r['per_sample_summary'] for r in results if r['Population'] == pop][0]
        print(f"Population {pop} (row {i}):")
        print(per_sample.round(3))
        print("-")

    print(f"\nPopulations with \\u2265 {min_cells_per_pop} cells but skipped due to insufficient per-sample coverage: {skipped_insufficient_samples}")

Number of Populations with \u2265 300 cells: 27
Large Populations: ['PA', 'PB', 'PC', 'PD', 'PE', 'PF', 'PG', 'PH', 'PI', 'PJ', 'PK', 'PL', 'PM', 'PN', 'PO', 'PP', 'PQ', 'PR', 'PS', 'PT', 'PU', 'PV', 'PW', 'PX', 'PY', 'PZ', 'PAA']



Per-Population Kruskal-Wallis results across Sample_ID (sorted by min FDR):
Population  n_total_cells  n_samples_tested  Complexity_H  Complexity_pval  Complexity_fdr  Complexity_eta2_approx  Purity_H  Purity_pval  Purity_fdr  Purity_eta2_approx    min_fdr
        PA          30380                 3     3.217e+03        0.000e+00       0.000e+00               1.059e-01 3.477e+02    3.137e-76   1.210e-75           1.145e-02  0.000e+00
        PR           4673                 3     2.183e+03        0.000e+00       0.000e+00               4.673e-01 1.747e+03    0.000e+00   0.000e+00           3.739e-01  0.000e+00
        PJ           9488                 3     1.284e+02        1.296e-28       1.842e-28               1.354e-02 1.427e+03   1.105e-310  1.492e-309           1.505e-01 1.492e-309
        PG          11596                 3     1.035e+03       2.117e-225      1.905e-224               8.923e-02 7.065e+01    4.557e-16   6.836e-16           6.093e-03 1.905e-224
        PB        

### Agent Interpretation

These results strongly support the core premise of the hypothesis: within fixed Populations, both Complexity and Purity show highly significant, structured variation across Sample_ID. The Kruskal–Wallis p-values and approximate effect sizes indicate that these shifts are not trivial and, in several cases, appear biologically interpretable rather than just noise.

Key points and how they should guide the next steps:

1. **Many Populations show robust across-sample shifts (candidate maturation axes).**  
   - Almost all large Populations have extremely small FDRs for at least one of Complexity or Purity; `min_fdr` is essentially 0 for the top ~half of the list. This fulfills the “FDR < 0.05 + non-trivial effect size” criterion for Step 2.  
   - Effect-size heuristics (KW eta²) suggest particularly strong sample-structured variation in some Populations:  
     - **PR:** Complexity_eta2 ≈ 0.47, Purity_eta2 ≈ 0.37 → very large across-sample shifts.  
     - **PV:** Complexity_eta2 ≈ 0.11, Purity_eta2 ≈ 0.23.  
     - **PA, PG, PQ, PX, PB, PC** also show notable eta² (0.05–0.11 range) in at least one metric.  
   These are ideal candidates for downstream maturation-signature derivation.

2. **Complexity and Purity are sometimes tightly coupled, sometimes decoupled across samples.**  
   From the per-sample summaries for the top 5:

   - **PA:**  
     - Complexity_mean: R77_4C4 > R78_4C12 > R78_4C15 (11.27 → 10.37 → 9.74).  
     - Purity_mean: R77_4C4 < R78_4C12 < R78_4C15 (0.455 → 0.463 → 0.479).  
     → As Complexity decreases across samples, Purity increases. This suggests an inverse relationship (more complex = “less pure”) within this Population, which is an interesting candidate for a maturation vs contamination/heterogeneity tradeoff.

   - **PR (very strong candidate):**  
     - Complexity_mean: R77_4C4 ≈ 12.39, R78_4C12 ≈ 12.11, R78_4C15 ≈ 6.86.  
     - Purity_mean: 0.421 → 0.523 → 0.738.  
     → Massive drop in Complexity and simultaneous large increase in Purity in R78_4C15. This looks like a clean, monotone and joint shift: PR in R78_4C15 is “low-complexity / high-purity” relative to the other samples. Very compelling as a maturation axis or state shift.

   - **PB (candidate where Complexity and Purity both change but less dramatically):**  
     - Complexity_mean: 6.06 (R77_4C4) → 6.76 (R78_4C12) → 5.52 (R78_4C15).  
     - Purity_mean: 0.685 → 0.704 → 0.705.  
     → Complexity is non-monotone; Purity increases modestly and then plateaus. Differences are smaller but still highly significant (huge N). This might reflect subtler or more complex maturation patterns.

   - **PG:**  
     - Complexity_mean: R78_4C12 > R77_4C4 > R78_4C15 (9.91 → 9.22 → 7.60).  
     - Purity_mean: slight non-monotone shifts (0.648 → 0.617 → 0.636).  
     → Strong Complexity shifts; Purity differences are smaller and do not show a clear monotone pattern, suggesting partial decoupling of these two metrics.

   - **PJ:**  
     - Complexity is very similar across samples (means 10.74–11.28), eta² is low (~0.0135).  
     - Purity varies strongly (H=1427, eta²≈0.15) with R78_4C15 > R77_4C4 > R78_4C12.  
     → Here Purity varies strongly while Complexity is almost “flat”. This is another useful scenario for testing whether transcriptional maturation can be captured primarily through Purity-linked signatures.

   These examples show that your hypothesis about “jointly structured shifts in Complexity and Purity” is partially validated: some Populations (especially PR, PA) show clearly structured, coordinated changes, while others show strong shifts mainly in one metric. That heterogeneity itself is informative.

3. **Which Populations look best for Step 2 (candidate selection)?**  
   Based on significance + effect size + interpretable trends from the summaries:

   - **High-priority “joint-shift” candidates** (clear, structured changes in both metrics, good cell counts):  
     - **PR:** textbook case; huge, monotone Complexity drop and Purity increase across samples.  
     - **PA:** smooth opposite trends in Complexity and Purity (lower Complexity, higher Purity across samples).  
     - **PB, PG, PQ, PC, PB, PN, PV, PX:** KW stats and eta² suggest robust shifts, though you’ll need to inspect per-sample summaries for each to confirm whether the trends are monotone or more complex.

   - **Purity-dominant with modest Complexity shifts (to test “Purity-only” maturation)**:  
     - **PJ, PK, PL, PZ, PY, PW, PAA** (high Purity H and eta², lower Complexity eta²) are useful to see whether gene-level maturation programs align more tightly with Purity than with Complexity.

   - **Complexity-dominant / weaker Purity shifts:**  
     - Some (like PH, PE) show much stronger KW statistics on Complexity than on Purity, indicating another axis where Complexity might be picking up biological changes not fully tracked by Purity.

4. **Implications for Step 3 (within-sample Complexity tertiles and DE):**

   For each chosen Population, the patterns above suggest specific strategies:

   - **PR and PA (and similar):**  
     - Within each Sample_ID, define Complexity tertiles and perform DE between top vs bottom tertiles as planned.  
     - **Critically, track within-sample Purity distributions across those tertiles.** For example:  
       - In PR, does R78_4C15 still show an internal Complexity–Purity relationship (e.g., low-complexity cells also higher Purity within that sample)? Or is the shift mostly between samples?  
     - After meta-analyzing DE across samples, check whether the “high-complexity” gene signature is enriched in the samples with higher mean Complexity and lower Purity (e.g., R77_4C4, R78_4C12 for PR).

   - **PJ and other “Purity-strong / Complexity-weak” populations:**  
     - It may be more informative here to mirror the same pipeline but using **Purity tertiles** instead of Complexity tertiles (or in parallel) to see if Purity defines clearer within-sample states.  
     - You can still stick to the original plan for consistency (Complexity tertiles) but explicitly test whether the DE and meta-analytic signatures also correlate with Purity within and across samples.

   - **Non-monotone patterns (e.g., PG, PB):**  
     - Don’t require strict monotonicity for inclusion; instead, record the direction per sample. For instance, PG has high Complexity in R78_4C12 and low in R78_4C15. When meta-analyzing, you might consider:  
       - Either ignoring direction and focusing on within-sample tertiles (as planned), or  
       - Encoding sample-level “Complexity rank” and checking whether effect directions align with that ordering.

5. **Refinements and checks before moving forward:**

   - **Visual inspection of distributions:**  
     For your shortlisted Populations (e.g., PR, PA, PB, PG, PJ, PV, PX), plot per-sample violin/boxplots for Complexity and Purity. This will verify that differences aren’t driven by bizarre outliers or bimodality that could indicate technical artifacts rather than maturation.

   - **Check Complexity–Purity correlation within each Pop × Sample:**  
     - Compute, for each (Population, Sample_ID), the correlation between Complexity and Purity.  
     - This will tell you whether within-sample maturation states tend to be low-complexity/high-purity (as suggested for PR) or something more nuanced. That information will help interpret DE patterns later and decide whether to stratify on Complexity, Purity, or both.

   - **Sample sizes per tertile:**  
     - The large populations (e.g., PA: ~10k cells per sample; PR: ~1k–2.6k per sample) will give very stable tertiles and DE results. For smaller ones (e.g., PX, PV), just confirm tertiles are still > ~50–100 cells each per sample to keep DE robust.

6. **How this supports the hypothesis so far:**

   - There is overwhelming evidence that within defined Populations, both Complexity and Purity are not constant across Sample_ID; instead, they show statistically and practically meaningful shifts.  
   - In several key Populations (notably PR, PA), Complexity and Purity change together in structured ways across samples, consistent with region-specific or time-specific maturation states that persist even after conditioning on cell identity.  
   - The diversity of patterns (joint, decoupled, Purity-only, Complexity-only) sets you up nicely to test whether molecular maturation signatures (from DE/meta-analysis) map consistently onto these metrics and onto spatial structure in later steps.

Concrete next actions:

- Select a **focused set of candidate Populations** for Step 3, e.g.:  
  - Joint-shift: PR, PA, PB, PG, PQ, PC, PV, PX.  
  - Purity-dominant: PJ, PK, PL, PY, PZ, PAA.  
  - Complexity-dominant: PH, PE, PS, PF.  
- For each chosen Population:
  - Generate per-sample violins of Complexity and Purity.  
  - Compute within-sample Complexity (and optionally Purity) tertiles and their Purity/Complexity distributions.  
  - Proceed with the planned within-sample DE (top vs bottom Complexity tertiles) and meta-analysis, explicitly annotating whether per-sample mean shifts match the directions of the DE-based maturation signature.

These steps will directly extend the promising patterns you’ve uncovered into gene-level maturation programs and set up the later spatial analyses.

## Next Steps
Step 1: Using the per-Population Kruskal–Wallis results, select a focused set of candidate Populations that show robust, interpretable across-sample shifts in Complexity and/or Purity (e.g., FDR < 0.05, non-trivial effect sizes, and coherent trends in per-sample means), explicitly annotating for each whether Complexity and Purity shifts are joint, Complexity-dominant, or Purity-dominant and ensuring adequate per-sample cell counts.
Step 2: Within each selected candidate Population, define per-sample Complexity tertiles (and record Purity distributions across these tertiles), then perform within-sample differential expression between highest and lowest Complexity tertiles and meta-analyze gene-level statistics across samples to obtain robust signatures of high- vs low-Complexity states.
Step 3: From the DE results, derive population-specific maturation gene signatures (e.g., top genes up in high- vs low-Complexity within each Population), score each cell with these signatures using sc.tl.score_genes, and formally test whether these scores, together with Complexity and Purity, show consistent shifts across Sample_ID within each Population using non-parametric tests while reporting effect sizes.
Step 4: For each key Population and Sample_ID, quantify spatial structuring of maturation scores and QC metrics by constructing a k-nearest-neighbor graph in spatial coordinates and computing Moran’s I per (Population, Sample_ID), summarizing which samples and spatial neighborhoods are enriched for high-maturation states and how this relates to Complexity/Purity shifts.

## This code refines the selection of candidate Populations with robust across-sample Complexity/Purity shifts by preserving the intended priority ordering, using an explicit biological sample order, and adding Spearman correlation-based trend summaries without using them as hard filters.

In [ ]:
import numpy as np
import pandas as pd

# We assume `res_df` and `results` exist from the previous Kruskal–Wallis step.
# If they do not, raise an error to enforce running step 1 first.
if 'res_df' not in globals() or 'results' not in globals():
    raise RuntimeError("Per-Population Kruskal–Wallis results (res_df, results) not found. Please run step 1 before step 2.")

# Copy to avoid accidental mutation
res_df_sel = res_df.copy().reset_index(drop=True)

# Define thresholds for robust across-sample shifts
fdr_threshold = 0.05
eta2_min_joint = 0.05    # heuristic: moderate effect size
eta2_min_single = 0.03   # heuristic: allow slightly smaller effects when only one metric is strong

# Annotate dominant metric category per population
categories = []
for _, row in res_df_sel.iterrows():
    e_c = row['Complexity_eta2_approx']
    e_p = row['Purity_eta2_approx']
    if (e_c >= eta2_min_joint) and (e_p >= eta2_min_joint):
        cat = 'joint'
    elif (e_c >= eta2_min_single) and (e_c > e_p):
        cat = 'complexity_dominant'
    elif (e_p >= eta2_min_single) and (e_p > e_c):
        cat = 'purity_dominant'
    else:
        cat = 'weak_or_ambiguous'
    categories.append(cat)

res_df_sel['shift_category'] = categories

# Compute minimum FDR and filter by FDR and shift category
res_df_sel['min_fdr'] = res_df_sel[['Complexity_fdr', 'Purity_fdr']].min(axis=1)

mask_signif = res_df_sel['min_fdr'] <= fdr_threshold
mask_category = res_df_sel['shift_category'] != 'weak_or_ambiguous'

candidates_df = res_df_sel.loc[mask_signif & mask_category].copy()

# Rank candidates: prioritize joint > complexity_dominant > purity_dominant, then by min_fdr, then by effect size sum
category_rank = {'joint': 0, 'complexity_dominant': 1, 'purity_dominant': 2}

candidates_df['category_rank'] = candidates_df['shift_category'].map(category_rank)
candidates_df['effect_sum'] = candidates_df['Complexity_eta2_approx'] + candidates_df['Purity_eta2_approx']

candidates_df = candidates_df.sort_values(
    by=['category_rank', 'min_fdr', 'effect_sum'],
    ascending=[True, True, False]
).reset_index(drop=True)

# Attach per-sample summaries from `results` and assess per-sample coverage & directionality
candidate_records = []
min_cells_per_sample = 200  # ensure we have enough cells per sample for tertiles & DE

# Optional: define an explicit biological sample order if known; otherwise fall back to lexicographic order
sample_order = ['R77_4C4', 'R78_4C12', 'R78_4C15']

for _, row in candidates_df.iterrows():
    pop = row['Population']
    # Find corresponding per-sample summary
    per_sample_summary = None
    for r in results:
        if r['Population'] == pop:
            per_sample_summary = r['per_sample_summary']
            break
    if per_sample_summary is None:
        continue

    # Check per-sample coverage
    n_cells_per_sample = per_sample_summary['n_cells']
    enough_cells_mask = n_cells_per_sample >= min_cells_per_sample
    n_samples_ok = int(enough_cells_mask.sum())

    # Skip populations that do not have at least 2 adequately powered samples
    if n_samples_ok < 2:
        continue

    # Order samples by predefined biological order where possible, otherwise by index
    available_samples = per_sample_summary.index.tolist()
    ordered_idx = [s for s in sample_order if s in available_samples]
    remaining = sorted([s for s in available_samples if s not in sample_order])
    ordered_idx.extend(remaining)
    ordered = per_sample_summary.loc[ordered_idx]

    comp_means = ordered['complexity_mean'].values
    purity_means = ordered['purity_mean'].values

    # Directionality summaries: strict monotonic flags plus a more flexible Spearman correlation
    def direction_flag(vals):
        diffs = np.diff(vals)
        if np.all(diffs > 0):
            return 1
        elif np.all(diffs < 0):
            return -1
        else:
            return 0

    def spearman_trend(vals):
        if len(vals) < 2:
            return np.nan
        # Rank by sample order 0..n-1
        ranks = np.arange(len(vals))
        rho, _ = stats.spearmanr(ranks, vals)
        return rho

    dir_complexity = direction_flag(comp_means)
    dir_purity = direction_flag(purity_means)
    rho_complexity = spearman_trend(comp_means)
    rho_purity = spearman_trend(purity_means)

    candidate_records.append({
        'Population': pop,
        'shift_category': row['shift_category'],
        'min_fdr': row['min_fdr'],
        'Complexity_eta2_approx': row['Complexity_eta2_approx'],
        'Purity_eta2_approx': row['Purity_eta2_approx'],
        'n_total_cells': row['n_total_cells'],
        'n_samples_with_>=min_cells': n_samples_ok,
        'samples_index_order': list(ordered.index),
        'complexity_means_ordered': comp_means,
        'purity_means_ordered': purity_means,
        'complexity_trend_flag': dir_complexity,
        'purity_trend_flag': dir_purity,
        'complexity_spearman_rho': rho_complexity,
        'purity_spearman_rho': rho_purity
    })

candidates_summary = pd.DataFrame(candidate_records)

if candidates_summary.empty:
    print("No candidate Populations passed FDR, effect size, and coverage criteria under current thresholds.")
else:
    # Rank final candidates for reporting, preserving categorical priority
    candidates_summary['category_rank'] = candidates_summary['shift_category'].map(category_rank)
    candidates_summary = candidates_summary.sort_values(
        by=['category_rank', 'min_fdr', 'Complexity_eta2_approx', 'Purity_eta2_approx'],
        ascending=[True, True, False, False]
    ).reset_index(drop=True)

    print("\nSelected candidate Populations for maturation-focused analysis (ranked):")
    cols_show = [
        'Population', 'shift_category', 'min_fdr',
        'Complexity_eta2_approx', 'Purity_eta2_approx',
        'n_total_cells', 'n_samples_with_>=min_cells',
        'complexity_trend_flag', 'purity_trend_flag',
        'complexity_spearman_rho', 'purity_spearman_rho'
    ]
    print(candidates_summary[cols_show].to_string(index=False, float_format=lambda x: f"{x:.3e}"))

    print("\nInterpretation of trend flags: +1 = strictly increasing across samples (in biological order), -1 = strictly decreasing, 0 = non-monotone. Spearman rho summarizes overall trend consistency without requiring strict monotonicity.")

    # For transparency, also print per-sample means for the top few candidates
    top_k = min(5, candidates_summary.shape[0])
    print(f"\nPer-sample Complexity and Purity means for top {top_k} candidate Populations:\n")
    for i in range(top_k):
        pop = candidates_summary.loc[i, 'Population']
        # Retrieve per-sample summary again
        per_sample_summary = None
        for r in results:
            if r['Population'] == pop:
                per_sample_summary = r['per_sample_summary']
                break
        print(f"Population {pop}:")
        print(per_sample_summary[['n_cells', 'complexity_mean', 'purity_mean']].round(3))
        print("-")

    # Store the final list of candidate Populations in a variable for downstream steps
    candidate_populations = candidates_summary['Population'].tolist()
    print("\nFinal candidate Populations to carry forward:", candidate_populations)



Selected candidate Populations for maturation-focused analysis (ranked):
Population      shift_category    min_fdr  Complexity_eta2_approx  Purity_eta2_approx  n_total_cells  n_samples_with_>=min_cells  complexity_trend_flag  purity_trend_flag  complexity_spearman_rho  purity_spearman_rho
        PR               joint  0.000e+00               4.673e-01           3.739e-01           4673                           3                     -1                  1               -1.000e+00            1.000e+00
        PV               joint 8.442e-115               1.053e-01           2.290e-01           2282                           2                      0                  0                      NaN                  NaN
        PQ               joint  3.253e-95               8.066e-02           7.867e-02           5429                           3                      0                 -1               -5.000e-01           -1.000e+00
        PX               joint  1.857e-27               7.

### Agent Interpretation

The selection step is doing exactly what you need for the hypothesis: you’ve now got a non-trivial, interpretable set of populations with strong across-sample shifts in Complexity and/or Purity and basic directional characterization. A few specific points and suggestions for how this informs and should shape the next steps:

1. **The hypothesis is provisionally supported at the metric level.**  
   - Many populations (PR, PA, PG, PB, etc.) show large η² for Complexity and/or Purity and very small FDRs, so there are definitely robust across-sample shifts.
   - Several populations show **coherent, monotone trends** across the three samples:  
     - Complexity strictly decreases from R77_4C4 → R78_4C15 (trend_flag = -1, Spearman = -1.0) in multiple populations (PR, PA, PC, PD, PM, PP, PT, PY, PK, PO, etc.).  
     - Purity often shows the **opposite or parallel monotone pattern** (e.g., PR, PA, PM, PT, PY, PK, PO have purity_trend_flag = +1, Spearman = 1.0).
   - This pattern—cells becoming “simpler” (lower Complexity) and “cleaner” (higher Purity) across samples—is very consistent with a maturation-like trajectory in several populations, so your selection criteria are indeed isolating plausible maturation candidates.

2. **Which populations look most promising for maturation trajectories?**  
   For moving into within-population DE and signature derivation, I’d prioritize those with:
   - ≥3 samples with ≥200 cells (already enforced), and  
   - Strong monotone or near-monotone trends in at least Complexity, ideally with aligned or interpretable Purity trends.

   From your table, especially promising are:

   **Joint shifts (Complexity + Purity):**
   - **PR**  
     - joint, massive η² (C ~0.47, P ~0.37) and FDR ~0.  
     - Complexity: strictly decreasing (12.39 → 12.11 → 6.86).  
     - Purity: strictly increasing (0.421 → 0.523 → 0.738).  
     - Very high n_total_cells (4673) and good coverage in all three samples.  
     - This is almost an “ideal” maturation-like pattern to model.
   - **PQ**  
     - joint, η²(C) ~0.08, η²(P) ~0.08; strong significance.  
     - Complexity: non-monotone but roughly “high–higher–lower” (10.65 → 11.08 → 9.54), Spearman -0.5.  
     - Purity: strictly decreasing across samples (0.524 → 0.521 → 0.443, flag -1, rho -1.0).  
     - This is interesting in that purity decreases while complexity is highest in the middle sample; suggests more nuanced state shifts, not a simple linear maturation.

   **Complexity-dominant:**
   - **PA**  
     - η²(C) ~0.11 is big; η²(P) small but non-zero.  
     - Complexity strictly decreases across samples; Purity strictly increases.  
     - Huge n_total_cells (30k+) and excellent coverage per sample.  
     - Great candidate for robust within-sample tertile DE and for detecting subtle gene-level maturation signatures.
   - **PM, PT, PY**  
     - All show strict monotone decrease in Complexity and strict increase in Purity (trend_flags -1 and +1; rhos ±1.0).  
     - Effect sizes are moderate; coverage good.  
     - These provide replicate examples of the same qualitative trajectory pattern in different populations—useful to test whether maturation signatures are population-specific vs. shared.
   - **PC, PD, PP**  
     - Strong effect sizes and clean monotone decrease in Complexity (trend_flag -1).  
     - Purity trends are weaker/ambiguous (trend_flag 0; rho 0.5) but still show some directional tendency (generally increasing).

   **Purity-dominant:**
   - **PK, PO**  
     - η²(P) substantial with monotone Purity increase and Complexity decrease.  
     - These look conceptually similar to PR/PA but with Purity driving the variance, so they’re ideal to test the hypothesis’ “Complexity vs Purity dominance” axis.
   - **PJ, PL, PAA**  
     - Purity-dominant but with weaker or non-monotone trend flags (rho ±0.5).  
     - Nice secondary candidates if you want more diverse behavior, but less “clean” than PR/PA/PM/PT/PY/PK/PO.

   **Borderline on coverage/monotonicity:**
   - **PV, PX, PW**  
     - Have only 2 samples with ≥200 cells; Spearman rhos are NaN because the third sample is essentially empty or excluded.  
     - They may still be biologically interesting but will give you less power and noisier tertile DE, especially PV (no cells in R78_4C15).

   **Actionable suggestion:** For the next DE step, I’d prioritize a core panel such as:  
   `['PR', 'PA', 'PM', 'PT', 'PY', 'PK', 'PO']`  
   and then optionally include `PQ, PC, PD, PP, PJ, PL, PAA` as secondary/contrast populations. PV, PX, PW could be held back for sensitivity analyses, given the weaker sample coverage.

3. **Interpreting joint vs. dominant vs. ambiguous patterns for downstream design.**  
   Your explicit shift_category annotation will be very helpful to stratify the maturation analysis:

   - **Joint populations (PR, PQ, PV, PX):**  
     - For these, you should carefully track whether gene-level maturation signatures correlate with both metrics or preferentially with one.  
     - When you compute maturation scores later, you can explicitly test whether within each Population both Complexity and Purity shift with those scores, or whether one metric is more tightly linked to the gene expression signature.
   - **Complexity-dominant (PA, PG, PB, PC, PD, PM, PP, PS, PT, PY, PZ):**  
     - For these, the DE and gene signatures should primarily be interpreted as Complexity-based states; Purity might reflect technical or secondary biological phenomena.  
     - In the modeling of gene signature scores vs. metrics, treat Purity more as a covariate/check rather than the main axis.
   - **Purity-dominant (PJ, PK, PL, PO, PW, PAA):**  
     - These are extremely useful to test whether there exist “maturation-like” shifts driven mostly by Purity rather than Complexity.  
     - This also helps ensure your downstream interpretation does not conflate general QC changes with maturation per se.

4. **Some code/analysis refinements before moving on:**

   - **Explicitly record the directionality (increasing vs decreasing) as a label.**  
     - You now have trend_flag and Spearman ρ, but for downstream steps, it will be extremely useful to store something like:
       - `complexity_direction = 'increase'/'decrease'/'non_monotone'`
       - `purity_direction = same`
     - This will make it easier, later on, to interpret whether “high Complexity tertile” corresponds to the earlier or later sample and how that maps onto presumed maturation stage.
   - **Consider relaxing the strict monotone requirement for some populations.**  
     - With only 3 samples, ANY slight non-linear pattern gives trend_flag = 0. Your use of Spearman ρ is good; you might want to classify populations with |ρ| ≥ 0.5 as “weak but overall increasing/decreasing” even when strict monotonicity fails.  
     - This could be turned into an additional annotation column, e.g., `complexity_trend_class = {'strong_inc','strong_dec','weak_inc','weak_dec','none'}` based on flag + ρ.
   - **Double-check the sample ordering assumption.**  
     - You’ve hard-coded `sample_order = ['R77_4C4', 'R78_4C12', 'R78_4C15']` as “biological order”. If biological time ordering is actually different, trend interpretations will invert.  
     - Before interpreting monotone directions as “maturation forward”, verify that this is the intended temporal/spatial progression; if not certain, be careful to use neutral language (“across-sample shift”) in the writeup and treat directionality more cautiously.
   - **Store per-population, per-sample summaries in a structured object.**  
     - You currently re-query `results` for each population when printing. It might be helpful to create a dictionary keyed by Population with the per_sample_summary and your computed trend metrics; you’ll reuse this in the tertile DE step to check per-sample coverage and distributions quickly.

5. **How this guides the upcoming DE and signature steps:**

   Given this selection, here’s how I’d align the next steps with your hypothesis and keep results distinct from the paper / prior analyses:

   - **Within-sample tertiles in prioritized populations:**  
     - For each of the core candidates (e.g., PR, PA, PM, PT, PY, PK, PO):
       - Within each Sample_ID, compute Complexity tertiles only if that (Population, Sample_ID) has enough cells (your global ≥200 cutoff is already ensuring this but you could also exclude borderline cases per sample).
       - Explicitly record how Purity is distributed across these Complexity tertiles; this will tell you whether Complexity tertiles are strongly confounded with Purity in each population.
     - You may want to also compute **Purity tertiles** in the Purity-dominant populations in parallel, to generate “Purity-based” maturation candidate signatures.

   - **Meta-analysis of DE across samples:**  
     - Because you already have clear directionality across samples, you can stratify DE analyses to test:
       - Whether genes up in high-Complexity within early samples are consistently down in later samples, etc.  
       - Whether the same genes show reciprocal trends in Purity-dominant vs Complexity-dominant populations.
     - Use per-sample DE (high vs low tertiles) and then a meta-analysis (Stouffer’s Z, random-effects, etc.) to avoid a single sample dominating.

   - **Interpreting maturation signatures:**
     - For the joint populations like PR and PQ, check whether the derived gene signatures correlate more with Complexity, Purity, or both when you score all cells.
     - For Complexity-dominant PA/PM/PT/PY, look for genes with strong association to Complexity but minimal association to Purity; these would be good candidates for “internal transcriptomic complexity” maturation signatures.
     - For Purity-dominant PK/PO/PJ/PL/PAA, treat their top genes as candidate signatures of **purity-driven** or “cleanup” processes, and check if they show similar spatial clustering as in Complexity-dominant populations or distinct patterns.

6. **Thinking ahead to spatial structuring:**

   Once you have maturation signatures, PR, PA, PM, PT, PY, PK, and PO are high-value for Moran’s I and spatial neighborhood analyses because:

   - They have large cell numbers and clear metric trends, making spatial patterns of maturation score and QC metrics easier to detect.
   - You can explicitly ask whether higher “maturation score” regions spatially co-localize with regions of higher Purity and/or lower Complexity, and whether that pattern is consistent across multiple populations that share similar shift categories.

   To stay distinct from typical heart spatial analyses (which often focus on anatomical region boundaries per se), emphasize:
   - Comparing spatial structuring of maturation scores **within the same cell population** across samples;  
   - Contrasting Complexity-dominant vs Purity-dominant cell types in terms of spatial clustering of matured states, rather than just raw expression of known developmental markers.

In summary, your candidate selection is working well and gives you a rich, interpretable set of populations that exhibit robust, directionally coherent shifts in Complexity and/or Purity. Focus the next DE and signature steps on the core monotone, well-powered populations (PR, PA, PM, PT, PY, PK, PO), explicitly leverage the shift_category and trend metrics in your design, and use Complexity vs Purity dominance as a central organizing axis when interpreting maturation signatures and their spatial organization.

## Next Steps
Step 1: Within each selected candidate Population (e.g., PR, PA, PM, PT, PY, PK, PO), restrict to Sample_IDs with adequate cell counts, define per-sample Complexity tertiles, verify that both high and low tertiles are well-populated, and summarize how Purity and UMI Count differ across these tertiles to understand their coupling to Complexity within each (Population, Sample_ID).
Step 2: For each candidate Population, perform within-sample differential expression between highest and lowest Complexity tertiles using a non-parametric Wilcoxon test (via sc.tl.rank_genes_groups with groupby on a local tertile label), then meta-analyze gene-level statistics across samples for that Population using a simple Stouffer Z-score combination to derive robust population-specific signatures of high- vs low-Complexity states, explicitly tracking how Complexity–Purity–UMI correlations may confound these signatures.
Step 3: From the population-level DE meta-analysis results, construct conservative gene signature tables for each Population (e.g., top genes up in high- vs low-Complexity by combined Z-score, consistent effect direction, and FDR), and print these signatures along with summary statistics indicating how strongly they align with Complexity vs Purity differences across samples, preparing them for later scoring and spatial analyses without performing those steps here.

## This code implements step 1 by defining per-sample Complexity tertiles within each selected candidate Population, assigning a local tertile label to cells that meet minimum count thresholds, and summarizing how Purity and UMI Count vary across these tertiles along with Complexity–Purity and Complexity–UMI correlations for each (Population, Sample_ID) pair.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats

# ------------------------------------------------------------------
# 1. Define the core candidate Populations based on prior KW/FDR step
# ------------------------------------------------------------------
# We assume `candidate_populations` and `candidates_summary` were
# created in the previous selection step. If not, we reconstruct a
# reasonable core set from `res_df` using the recorded shift_category
# and trend metrics. This keeps the analysis self-contained.

if 'candidate_populations' in globals():
    core_candidates = candidate_populations
else:
    if 'res_df' not in globals():
        raise RuntimeError("res_df with per-Population KW results not found; run the KW step before this analysis.")
    # Heuristic reconstruction: prioritize strongly significant, clear-shift populations
    tmp = res_df.copy()
    tmp['min_fdr'] = tmp[['Complexity_fdr', 'Purity_fdr']].min(axis=1)
    # Require fairly strong effects on at least one metric
    tmp = tmp[(tmp['min_fdr'] <= 0.05) & ((tmp['Complexity_eta2_approx'] >= 0.05) | (tmp['Purity_eta2_approx'] >= 0.05))]
    # Take up to 7 largest by effect size sum as a fallback core set
    tmp['effect_sum'] = tmp['Complexity_eta2_approx'] + tmp['Purity_eta2_approx']
    core_candidates = tmp.sort_values('effect_sum', ascending=False)['Population'].head(7).tolist()

print("Core candidate Populations to analyze:", core_candidates)

# ---------------------------------------------------------------
# 2. For each candidate Population, define per-sample Complexity
#    tertiles and summarize Purity / UMI Count by tertile.
# ---------------------------------------------------------------

obs = adata.obs.copy()
required_cols = ['Populations', 'Sample_ID', 'Complexity', 'Purity', 'UMI Count']
missing = [c for c in required_cols if c not in obs.columns]
if missing:
    raise ValueError(f"Missing required columns in adata.obs: {missing}")

# Storage for later DE/meta-analysis
tertile_label_col = 'Complexity_tertile_local'
if tertile_label_col in obs.columns:
    # Avoid clobbering any existing column in the working copy
    obs = obs.drop(columns=[tertile_label_col])

# Parameters
min_cells_per_pop_sample = 150  # ensure robust tertiles and DE

summary_records = []

for pop in core_candidates:
    mask_pop = obs['Populations'] == pop
    df_pop = obs.loc[mask_pop, ['Sample_ID', 'Complexity', 'Purity', 'UMI Count']].copy()
    n_pop = df_pop.shape[0]
    if n_pop == 0:
        continue

    # Per-sample processing: define tertiles where enough cells
    sample_ids = df_pop['Sample_ID'].unique().tolist()
    for sid in sample_ids:
        mask_ps = (df_pop['Sample_ID'] == sid)
        df_ps = df_pop.loc[mask_ps].copy()
        n_ps = df_ps.shape[0]
        if n_ps < min_cells_per_pop_sample:
            # Not enough cells to form stable tertiles for this (Pop, Sample)
            continue

        # Compute Complexity tertile cutoffs (using quantiles 0, 1/3, 2/3, 1)
        qs = df_ps['Complexity'].quantile([0.0, 1/3, 2/3, 1.0]).values
        q1, q2 = qs[1], qs[2]

        # Assign local tertile labels within this (Population, Sample_ID)
        # 0 = low, 1 = mid, 2 = high
        comp_vals = df_ps['Complexity'].values
        tertile_labels = np.full_like(comp_vals, fill_value=-1, dtype=int)
        tertile_labels[comp_vals <= q1] = 0
        tertile_labels[(comp_vals > q1) & (comp_vals <= q2)] = 1
        tertile_labels[comp_vals > q2] = 2

        # Keep only if we have non-trivial low and high tertiles
        n_low = int((tertile_labels == 0).sum())
        n_high = int((tertile_labels == 2).sum())
        if n_low < 30 or n_high < 30:
            # Too few cells in an extreme tertile; skip this sample for DE
            continue

        # Write labels back into obs
        idx_ps = df_ps.index
        obs.loc[idx_ps[tertile_labels == 0], tertile_label_col] = 0
        obs.loc[idx_ps[tertile_labels == 1], tertile_label_col] = 1
        obs.loc[idx_ps[tertile_labels == 2], tertile_label_col] = 2

        # Summarize Complexity / Purity / UMI Count by tertile for this (Pop, Sample)
        df_ps[tertile_label_col] = tertile_labels
        summ = df_ps.groupby(tertile_label_col).agg(
            n_cells=('Complexity', 'size'),
            complexity_mean=('Complexity', 'mean'),
            complexity_median=('Complexity', 'median'),
            purity_mean=('Purity', 'mean'),
            purity_median=('Purity', 'median'),
            umi_mean=('UMI Count', 'mean'),
            umi_median=('UMI Count', 'median')
        )

        # Compute within-sample Spearman correlations for Complexity vs Purity and vs UMI Count
        rho_cp, p_cp = stats.spearmanr(df_ps['Complexity'], df_ps['Purity'])
        rho_cu, p_cu = stats.spearmanr(df_ps['Complexity'], df_ps['UMI Count'])

        summary_records.append({
            'Population': pop,
            'Sample_ID': sid,
            'n_cells_used': n_ps,
            'n_low_tertile': n_low,
            'n_high_tertile': n_high,
            'complexity_tertile_stats': summ,
            'rho_Complexity_Purity': rho_cp,
            'pval_Complexity_Purity': p_cp,
            'rho_Complexity_UMI': rho_cu,
            'pval_Complexity_UMI': p_cu
        })

# Convert summary to a flat DataFrame for printing
if len(summary_records) == 0:
    print("No (Population, Sample_ID) pairs passed the cell-count criteria for tertile definition.")
    tertile_summary_df = pd.DataFrame()
else:
    flat_rows = []
    for rec in summary_records:
        # Extract simple scalar summaries for low (0) and high (2) tertiles
        stats_tbl = rec['complexity_tertile_stats']
        # Use get with default NaN to handle missing tertiles gracefully
        def get_stat(t, col):
            return stats_tbl.loc[t, col] if (t in stats_tbl.index) else np.nan
        flat_rows.append({
            'Population': rec['Population'],
            'Sample_ID': rec['Sample_ID'],
            'n_cells_used': rec['n_cells_used'],
            'n_low_tertile': rec['n_low_tertile'],
            'n_high_tertile': rec['n_high_tertile'],
            'low_complexity_mean': get_stat(0, 'complexity_mean'),
            'mid_complexity_mean': get_stat(1, 'complexity_mean'),
            'high_complexity_mean': get_stat(2, 'complexity_mean'),
            'low_purity_mean': get_stat(0, 'purity_mean'),
            'mid_purity_mean': get_stat(1, 'purity_mean'),
            'high_purity_mean': get_stat(2, 'purity_mean'),
            'low_umi_mean': get_stat(0, 'umi_mean'),
            'mid_umi_mean': get_stat(1, 'umi_mean'),
            'high_umi_mean': get_stat(2, 'umi_mean'),
            'rho_Complexity_Purity': rec['rho_Complexity_Purity'],
            'pval_Complexity_Purity': rec['pval_Complexity_Purity'],
            'rho_Complexity_UMI': rec['rho_Complexity_UMI'],
            'pval_Complexity_UMI': rec['pval_Complexity_UMI']
        })

    tertile_summary_df = pd.DataFrame(flat_rows)

    # Print an overview of how many (Pop, Sample) pairs are usable
    print("\nSummary of (Population, Sample_ID) pairs with usable Complexity tertiles:")
    print(tertile_summary_df[['Population', 'Sample_ID', 'n_cells_used', 'n_low_tertile', 'n_high_tertile']]
          .sort_values(['Population', 'Sample_ID'])
          .to_string(index=False))

    # For each candidate Population, print a concise summary of tertile-level means and correlations
    print("\nPer-Population detailed summaries of Complexity, Purity, and UMI Count across Complexity tertiles (low/mid/high):\n")
    for pop in core_candidates:
        df_pop_sum = tertile_summary_df[tertile_summary_df['Population'] == pop]
        if df_pop_sum.empty:
            print(f"Population {pop}: no samples met criteria for tertile analysis.\n")
            continue
        print(f"Population {pop}:")
        cols_show = [
            'Sample_ID', 'n_cells_used', 'n_low_tertile', 'n_high_tertile',
            'low_complexity_mean', 'mid_complexity_mean', 'high_complexity_mean',
            'low_purity_mean', 'mid_purity_mean', 'high_purity_mean',
            'low_umi_mean', 'mid_umi_mean', 'high_umi_mean',
            'rho_Complexity_Purity', 'pval_Complexity_Purity',
            'rho_Complexity_UMI', 'pval_Complexity_UMI'
        ]
        print(df_pop_sum[cols_show].round(3).to_string(index=False))
        print("-")

# Finally, write the tertile labels back into the AnnData object for downstream DE.
# If no tertiles were assigned, this will create a column of NaNs.
adata.obs[tertile_label_col] = obs.get(tertile_label_col, np.nan)

# Optionally store the flat summary for downstream interpretation
complexity_tertiles_summary = tertile_summary_df

print(f"\nStored per-(Population, Sample_ID) Complexity tertile labels in adata.obs['{tertile_label_col}'], and saved summary in `complexity_tertiles_summary`.")


Core candidate Populations to analyze: ['PR', 'PV', 'PQ', 'PX', 'PA', 'PG', 'PB', 'PC', 'PD', 'PM', 'PP', 'PS', 'PT', 'PY', 'PZ', 'PJ', 'PK', 'PL', 'PO', 'PW', 'PAA']



Summary of (Population, Sample_ID) pairs with usable Complexity tertiles:
Population Sample_ID  n_cells_used  n_low_tertile  n_high_tertile
        PA   R77_4C4          8718           2985            1953
        PA  R78_4C12         10008           3695            2605
        PA  R78_4C15         11654           5356            3602
       PAA   R77_4C4           413            170             100
       PAA  R78_4C12           392            133             106
       PAA  R78_4C15           222             98              46
        PB   R77_4C4          5945           2697            1287
        PB  R78_4C12          7755           2766            1801
        PB  R78_4C15          6247           2545            1577
        PC   R77_4C4          5074           1993             934
        PC  R78_4C12          5988           2139            1511
        PC  R78_4C15          6522           2550            1206
        PD   R77_4C4          4817           1810            1095
 

### Agent Interpretation

The tertile construction step worked very cleanly and sets you up well for the within-sample DE/meta-analysis. A few points that are especially informative for the hypothesis and how to proceed:

1. **You have ample usable (Population, Sample) pairs for almost all candidates.**  
   - Most core populations have 1.5–10k cells per sample with hundreds of cells in both low and high tertiles (often >400 per extreme tertile).  
   - This is ideal for robust within-sample Wilcoxon DE and for a Stouffer meta-analysis, and it means you can reasonably expect to detect subtle, shared transcriptional programs.

2. **Complexity tertiles are well-separated and monotonic.**  
   - For every population and sample shown, low < mid < high in `complexity_mean`, with substantial absolute separation (e.g. in PM, low ~5.5 vs high ~12.9, in PK, low ~10.4 vs high ~16.7).  
   - This confirms that your tertile labels capture a genuine within-population gradient, not noise, which is exactly what you need to test for “maturation-like” programs.

3. **Complexity–Purity coupling is strong and almost universally negative.**  
   - Spearman ρ(Complexity, Purity) is consistently negative, often strongly so (e.g., PM: –0.54 to –0.74, PG: –0.54 to –0.66, PX and PZ: ~–0.5 to –0.65, PR: up to –0.85).  
   - Tertile means reflect this: low-complexity cells often have higher Purity than high-complexity cells (sometimes modestly, sometimes dramatically).  
   - Interpretation: higher Complexity in this dataset tends to coincide with more “mixed” or less pure expression profiles, which could reflect developmental state, technical contamination, doublets, or more transitional cell states.  
   - For your hypothesis, this is both an opportunity and a confound: any DE associated with Complexity will almost certainly also be associated with Purity.

4. **Complexity–UMI relationships vary by population and are often modest.**  
   - ρ(Complexity, UMI Count) ranges from clearly negative (PB, PM, PS: –0.2 to –0.46) to near-zero or weakly positive in some cases (PG, PT in certain samples: 0.1–0.23).  
   - Within tertiles, `umi_mean` often decreases from low to high Complexity (e.g. PB, PM, PS, many others), but not uniformly (PT has low<mid<high UMIs in some samples).  
   - This suggests that Complexity is not simply a proxy for read depth and that your DE signals will not be trivially reducible to UMI differences, but you will need to remain aware of this coupling.

5. **Some populations show more dramatic and consistent patterns than others.**  
   These are particularly promising for detecting robust programs:

   - **PM, PB, PS, PX, PZ, PG, PA, PC, PD:**  
     - Large sample sizes, very clear low→mid→high Complexity gradients, and strong negative correlations with Purity.  
     - Often substantial shifts in UMI as well.  
     - These are prime candidates for strong, meta-analyzable DE signatures and will likely yield coherent “high Complexity” vs “low Complexity” gene sets.

   - **PK, PO, PW, PP, PJ, PL, PR, PV, PY, PAA:**  
     - Also have good cell numbers and clear Complexity gradients.  
     - Correlations with Purity are mostly negative, but magnitude is sometimes smaller (e.g., PJ, PL) or variable.  
     - Still valuable, but signatures may be more heterogeneous or more sensitive to confounds.

   I would prioritize the first group (PM/PB/PS/PX/PZ/PG/PA/PC/PD) for the initial meta-analysis and then see how well results generalize to the second group.

6. **Implications for the hypothesis (“maturation-like” signatures):**  
   - The within-population structure is strong enough that it is very plausible you’ll detect systematic gene-expression differences between low vs high Complexity tertiles.  
   - However, because high Complexity ≈ low Purity in most populations, any observed “maturation-like” program will need to be interpreted in the context of potential deconvolution/mixing effects, not just intrinsic maturation.  
   - It’s possible that what you call “maturation-like” is actually a gradient from more identity-pure, possibly more differentiated cells (high Purity, low Complexity) to more transcriptionally diverse, transitional, or microenvironmentally-influenced cells (high Complexity, low Purity).

7. **Concrete suggestions for the next DE/meta-analysis step:**

   **a. Stratify and annotate the confounding structure explicitly.**  
   For each (Population, Sample_ID), you already have:
   - ΔComplexity (high – low tertile means)  
   - ΔPurity  
   - ΔUMI  

   When you meta-analyze DE:
   - For each gene and population, alongside the combined Z-score, compute the correlation across samples between gene-level logFC and ΔComplexity, ΔPurity, and ΔUMI.  
   - This will let you tag genes as “Complexity-aligned but also strongly Purity-aligned” vs “Complexity-aligned with minimal Purity/UMI dependence.” Those latter genes are stronger candidates for intrinsic programs.

   **b. Consider minimal per-sample confound control.**  
   Staying within your current Wilcoxon plan but adding interpretive layers:
   - Within each sample, you might compute differential expression not just low vs high tertile, but also record per-group mean Purity and UMI, to use as covariates in downstream interpretation (not in the test itself, to keep it simple).  
   - Optionally, for the most promising populations, you could later re-run a small subset of genes using a generalized linear model (per-sample) adjusting for log(UMI) and/or Purity as covariates, to validate that your top Complexity-associated genes persist when controlling for obvious confounds.

   **c. Focus on consistency of direction across samples.**  
   - Require that a gene’s effect direction (up in high vs low Complexity) is consistent across most or all samples within a population to be included in the “robust” signature.  
   - For each gene, count how many samples show logFC in the same direction as the combined Z-score, and report this fraction.

   **d. Construct tiered gene sets per population.**  
   For each population, after meta-analysis, define at least three layers:
   - **Core Complexity signature:** genes with strong |Z|, FDR-corrected significance, consistent direction, and low correlation of logFC with ΔUMI and ΔPurity.  
   - **Complexity–Purity coupled signature:** genes strongly associated with both Complexity and Purity differences.  
   - **Ambiguous/technical candidates:** genes whose effect size appears driven largely by UMI or is inconsistent across samples.

   This tiering directly implements your goal of “tracking how Complexity–Purity–UMI correlations may confound these signatures.”

   **e. Compare signatures across populations (later step).**  
   - Once you’ve constructed per-population core signatures, evaluate overlap and correlation between populations to identify shared vs population-specific “high Complexity” programs, which is key for claiming a generalized “maturation-like” program versus population-specific state transitions.

8. **Quality checks before or in parallel with DE:**

   - For a few representative populations, it would be helpful to:
     - Plot distributions of Complexity, Purity, and UMI per tertile (e.g., violin or boxplots) to visualize the gradients.  
     - Confirm that spatial distributions of low vs high tertiles within a population are not trivially separable by batch/sample artifact (e.g., high Complexity only at the edge of tissue or in low-density areas). This can be deferred to later but will strengthen interpretation.

9. **How these results bear on the hypothesis right now:**

   - The current step does not yet test for transcriptional programs, but it strongly supports the **preconditions** for your hypothesis:
     - Within each candidate population and sample, Complexity forms a robust gradient with large dynamic range.  
     - There is strong, quantifiable coupling to Purity (and to a lesser extent UMI).  
   - This means that within-sample DE/meta-analysis is well-justified and likely to yield reproducible signatures.  
   - The main scientific tension to resolve is whether those signatures represent intrinsic maturation vs changes in mixing/contamination vs other state changes. Your explicit tracking of Complexity–Purity–UMI relationships is exactly what you’ll need to refine that interpretation.

In summary, proceed with the planned within-sample Wilcoxon DE and Stouffer meta-analysis, giving priority to high-count, strongly gradient populations (e.g., PM/PB/PS/PX/PZ/PG/PA/PC/PD), and systematically annotate how each gene’s signal relates to Purity and UMI differences across samples. That will put you in a strong position to extract conservative, interpretable “high vs low Complexity” signatures and to judge how maturation-like they truly are.

## Next Steps
Step 1: For each well-powered candidate Population prioritized from prior KW/FDR-based selection (e.g., intersecting core_pref with candidate_populations), restrict to (Population, Sample_ID) pairs flagged as usable in complexity_tertiles_summary, then perform within-sample differential expression between high (Complexity_tertile_local=2) and low (Complexity_tertile_local=0) Complexity cells using Wilcoxon tests via sc.tl.rank_genes_groups, extracting per-sample log fold-changes, p-values, and effect scores for all genes plus ΔComplexity, ΔPurity, and ΔUMI summaries.
Step 2: For each Population, meta-analyze per-gene statistics across contributing samples using a Stouffer Z-score combination signed by within-sample logFC direction, apply BH FDR correction, and construct conservative high- and low-Complexity gene signature tables that include combined Z, meta p/FDR, direction consistency across samples, and correlations between per-sample logFC and ΔPurity/ΔUMI, printing all results as text tables.

## This code refines the within-sample differential expression step by aligning candidate Population selection with prior KW/FDR-based candidates and the precomputed complexity_tertiles_summary, then performs Wilcoxon DE between high and low Complexity tertiles within each well-powered (Population, Sample_ID), storing per-sample gene-level statistics and ΔComplexity/ΔPurity/ΔUMI summaries for later meta-analysis.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import stats

# We assume `adata` is in memory and that prior steps have created:
# - per-cell Complexity tertiles in adata.obs['Complexity_tertile_local']
# - a summary table `complexity_tertiles_summary` with usable (Population, Sample_ID) pairs
# - a KW/FDR-based `candidate_populations` list prioritizing key Populations

# ------------------------------
# 1. Define candidate populations consistent with prior selection
# ------------------------------

core_pref = ['PA', 'PB', 'PC', 'PD', 'PG', 'PM', 'PS', 'PX', 'PZ']
obs = adata.obs

if 'candidate_populations' in globals():
    # Intersect preferred core list with previously selected candidates
    available_core = [p for p in core_pref if p in candidate_populations]
    if len(available_core) > 0:
        core_candidates = available_core
    else:
        # Fall back to all candidate_populations that actually exist in obs
        core_candidates = [p for p in candidate_populations if p in obs['Populations'].unique()]
else:
    # Fallback: intersect core_pref with observed Populations
    available_pops = set(obs['Populations'].unique().tolist())
    core_candidates = [p for p in core_pref if p in available_pops]

if len(core_candidates) == 0:
    # Final fallback: choose top 5 most abundant populations
    pop_counts = obs['Populations'].value_counts()
    core_candidates = pop_counts.head(5).index.tolist()

print("Candidate Populations for within-sample DE/meta-analysis:", core_candidates)

# Column with local Complexity tertiles
tertile_col = 'Complexity_tertile_local'
if tertile_col not in obs.columns:
    raise RuntimeError(f"Expected tertile labels in adata.obs['{tertile_col}'], but column is missing.")

# Ensure required covariate columns exist
required_cols = ['Populations', 'Sample_ID', 'Complexity', 'Purity', 'UMI Count', tertile_col]
missing = [c for c in required_cols if c not in obs.columns]
if missing:
    raise ValueError(f"Missing required columns in adata.obs: {missing}")

# ---------------------------------------------
# 2. Prepare containers for per-population DE
# ---------------------------------------------

meta_results = {}   # will hold per-Population meta-analysis inputs (filled in step 2)
per_sample_de = {}  # per (Population, Sample_ID) DE and summary tables

all_genes = adata.var_names.to_list()

# If available, restrict to (Population, Sample_ID) pairs deemed usable in the tertile summary
usable_pairs = None
if 'complexity_tertiles_summary' in globals() and isinstance(complexity_tertiles_summary, pd.DataFrame):
    usable_pairs = set(
        zip(
            complexity_tertiles_summary['Population'].astype(str),
            complexity_tertiles_summary['Sample_ID'].astype(str)
        )
    )

# Utility: run within-sample DE for a specific (Population, Sample_ID)

def run_within_sample_de(pop, sample_id):
    """Run Wilcoxon DE for high vs low Complexity tertiles within a single (Population, Sample_ID).
    Returns a DataFrame of genes with logFC, scores, pvals, and attached Population/Sample_ID.
    """
    # Optional: skip pairs not in the precomputed usable set
    if usable_pairs is not None and (pop, sample_id) not in usable_pairs:
        return None

    # Boolean mask for this population and sample and usable tertiles
    mask = (
        (adata.obs['Populations'] == pop) &
        (adata.obs['Sample_ID'] == sample_id) &
        (adata.obs[tertile_col].isin([0, 2]))
    )
    if mask.sum() < 60:
        return None  # not enough cells overall

    # Create a view for this subset
    ad_sub = adata[mask].copy()

    # Assert that data are log-normalized; if not, user must ensure upstream preprocessing
    if np.max(ad_sub.X) > 50:
        print(f"Warning: Expression values for (Population {pop}, Sample {sample_id}) appear large; ensure log-normalization was performed upstream.")

    # Build a local label: 'low' for tertile 0, 'high' for tertile 2
    tertiles_local = ad_sub.obs[tertile_col].astype(int)
    group_labels = np.where(tertiles_local == 2, 'high', 'low')
    ad_sub.obs['complexity_bin'] = pd.Categorical(group_labels, categories=['low', 'high'])

    # Require reasonable group sizes
    counts = ad_sub.obs['complexity_bin'].value_counts()
    if ('low' not in counts) or ('high' not in counts) or (counts['low'] < 30) or (counts['high'] < 30):
        return None

    # Run Wilcoxon DE: high vs low
    sc.tl.rank_genes_groups(
        ad_sub,
        groupby='complexity_bin',
        groups=['high'],
        reference='low',
        method='wilcoxon',
        use_raw=False,
        n_genes=ad_sub.n_vars,
        pts=True
    )

    rg = ad_sub.uns['rank_genes_groups']
    genes = rg['names']['high']
    scores = rg['scores']['high']
    pvals = rg['pvals']['high']
    pvals_adj = rg['pvals_adj']['high'] if 'pvals_adj' in rg else None
    logfc = rg['logfoldchanges']['high'] if 'logfoldchanges' in rg else None

    de_df = pd.DataFrame({
        'gene': genes,
        'score': scores,
        'pval': pvals
    })
    if pvals_adj is not None:
        de_df['pval_adj'] = pvals_adj
    else:
        de_df['pval_adj'] = np.nan
    if logfc is not None:
        de_df['logFC'] = logfc
    else:
        de_df['logFC'] = np.nan

    # Attach sample and population
    de_df['Population'] = pop
    de_df['Sample_ID'] = sample_id

    # Drop any genes with NA pvals before returning
    de_df = de_df.dropna(subset=['pval'])
    return de_df

# ---------------------------------------------------
# 3. Run within-sample DE for each candidate Population
# ---------------------------------------------------

for pop in core_candidates:
    print(f"\nRunning within-sample DE for Population {pop}...")

    # Identify samples where this Population has tertile-labeled cells
    mask_pop = (obs['Populations'] == pop) & (obs[tertile_col].isin([0, 2]))
    if not mask_pop.any():
        print(f"  No tertile-labeled cells for Population {pop}; skipping.")
        continue

    df_pop = obs.loc[mask_pop, ['Sample_ID', tertile_col, 'Purity', 'UMI Count', 'Complexity']]

    # Keep samples with enough cells in both extremes (consistent with tertile summary)
    sample_counts = df_pop['Sample_ID'].value_counts()
    eligible_samples = []
    for sid in sample_counts.index:
        df_ps = df_pop[df_pop['Sample_ID'] == sid]
        n_low = (df_ps[tertile_col] == 0).sum()
        n_high = (df_ps[tertile_col] == 2).sum()
        if (n_low >= 30) and (n_high >= 30):
            eligible_samples.append(sid)

    if len(eligible_samples) < 2:
        print(f"  Fewer than 2 eligible samples with adequate low/high tertiles for {pop}; skipping meta-analysis for this Population.")
        continue

    de_list = []
    per_sample_summaries = []

    for sid in eligible_samples:
        de_df = run_within_sample_de(pop, sid)
        if de_df is None:
            print(f"  Skipping (Population {pop}, Sample {sid}) due to insufficient cells or tertile structure.")
            continue

        de_list.append(de_df)

        # Also capture per-sample tertile-level metric differences (for confound tracking)
        df_ps = df_pop[df_pop['Sample_ID'] == sid].copy()
        low_mask = df_ps[tertile_col] == 0
        high_mask = df_ps[tertile_col] == 2
        if low_mask.sum() > 0 and high_mask.sum() > 0:
            d_complexity = df_ps.loc[high_mask, 'Complexity'].mean() - df_ps.loc[low_mask, 'Complexity'].mean()
            d_purity = df_ps.loc[high_mask, 'Purity'].mean() - df_ps.loc[low_mask, 'Purity'].mean()
            d_umi = df_ps.loc[high_mask, 'UMI Count'].mean() - df_ps.loc[low_mask, 'UMI Count'].mean()
        else:
            d_complexity = d_purity = d_umi = np.nan

        per_sample_summaries.append({
            'Population': pop,
            'Sample_ID': sid,
            'n_low': int(low_mask.sum()),
            'n_high': int(high_mask.sum()),
            'delta_Complexity_high_minus_low': d_complexity,
            'delta_Purity_high_minus_low': d_purity,
            'delta_UMI_high_minus_low': d_umi
        })

    if len(de_list) == 0:
        print(f"  No per-sample DE results for Population {pop}; skipping meta-analysis for this Population.")
        continue

    # Concatenate per-sample DE
    de_all = pd.concat(de_list, axis=0, ignore_index=True)
    per_sample_de[(pop, 'summary')] = pd.DataFrame(per_sample_summaries)

    # Store full DE table for this Population; meta-analysis will be performed in step 2
    per_sample_de[(pop, 'full_de')] = de_all

print("\nCompleted within-sample DE for candidate Populations. Per-sample DE results stored in `per_sample_de`. Meta-analysis will be performed in the next step.")


Candidate Populations for within-sample DE/meta-analysis: ['PA', 'PB', 'PC', 'PD', 'PG', 'PM', 'PS', 'PX', 'PZ']

Running within-sample DE for Population PA...
ranking genes


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


ranking genes


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


ranking genes


...

ranking genes


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


ranking genes


    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)



Completed within-sample DE for candidate Populations. Per-sample DE results stored in `per_sample_de`. Meta-analysis will be performed in the next step.


### Agent Interpretation

The current step did what it needed to: you now have reasonably powered, *within-sample* high- vs low-Complexity DE tables for most of the core cardiac populations, which is exactly what you need for the meta-analysis and for testing the hypothesis.

Key points and suggestions:

1. **Evidence that the hypothesis is testable from these results**
   - For PA, PB, PC, PD, PG, PM, PS, PX, PZ you successfully ran Wilcoxon DE in at least two samples with ≥30 cells per tertile-extreme. That means:
     - There is sufficient within-sample structure in Complexity for these populations.
     - You have per-gene logFC and p-values per sample, plus ΔComplexity/ΔPurity/ΔUMI per sample in `per_sample_de[(pop, 'summary')]`.
   - This is exactly the substrate needed to ask: “Are there genes consistently up in high-Complexity cells across samples within a population, and are these effects independent of/ coupled to Purity and UMI Count?”

2. **What to inspect now, before meta-analysis**
   Even though you haven’t printed tables yet, some quick QC on `per_sample_de` will help guide a conservative meta-analysis:
   - For a few populations (e.g. PA, PC, PS):
     - Inspect the distribution of logFCs and adjusted p-values per sample. Are there many genes with consistent direction and strong effect, or is it noisy?
     - Check the `per_sample_de[(pop, 'summary')]` table:
       - Are `delta_Complexity_high_minus_low` large and positive as expected?
       - Are `delta_Purity` and `delta_UMI` roughly centered near zero, or do some samples show large shifts? Those will be important when quantifying coupling later.

   If you find populations where ΔPurity or ΔUMI are systematically large whenever ΔComplexity is large, you’ll want to treat those populations with extra caution in the meta-analysis (or down-weight those samples).

3. **Strengths of the current implementation**
   - **Within-sample contrasts:** You are not pooling across samples; this reduces confounding by sample-level differences and aligns with the “within-sample transcriptional programs” aspect of the hypothesis.
   - **Filtering for tertile extremes with minimum n:** Requiring ≥30 cells per group and ≥2 samples per population makes the downstream meta-analysis more stable and less driven by single noisy samples.
   - **Capturing ΔComplexity, ΔPurity, ΔUMI per sample:** This is critical for the “coupled to Purity and UMI Count” part of the hypothesis and positions you to compute correlations between gene-level logFC and these deltas in the meta step.

4. **Potential improvements/edge cases to keep in mind**
   - **Usable-pair filtering:** You enforce `usable_pairs` from `complexity_tertiles_summary` only *inside* `run_within_sample_de`. In the outer loop, you re-derive eligible samples from counts, which could include samples not in `usable_pairs`. That’s fine as long as `usable_pairs` is consistent with your >30 threshold, but if `complexity_tertiles_summary` encoded other QC filters, you might want to also restrict `eligible_samples` to `Sample_ID`s present in `usable_pairs` for that population.
   - **Expression scale check:** You print a warning if `np.max(ad_sub.X) > 50`. It might be useful to confirm once that this condition is never hit; if it is, you should either re-run with proper log-normalization or subset to a log-normalized layer (e.g., `layer='log1p'`) before trusting DE.
   - **Missing logFC or pvals_adj:** You already guard against missing `logfoldchanges` and `pvals_adj`. If many genes have `NaN` logFC, that will weaken the meta-analysis. It’s worth confirming that `de_df['logFC']` is well-populated for a few populations.

5. **Recommendations for the upcoming meta-analysis step**
   To move toward “conservative maturation-like gene signatures” and explicitly assess coupling to Purity/UMI:

   - **Per-gene Stouffer meta-Z within each population:**
     - For each population `pop`, start from `de_all = per_sample_de[(pop, 'full_de')]`.
     - Pivot to get a matrix: rows = genes, columns = Sample_ID; entries = per-sample Z-scores or signed Z from Wilcoxon (you can convert from p-values using the sign of logFC: `Z = sign(logFC) * norm.isf(pval/2)`).
     - Filter genes that appear in *most* samples (e.g. expressed in ≥70% of samples for that population) to avoid single-sample artifacts.
     - Combine using Stouffer: `Z_meta = sum(Z_i * w_i) / sqrt(sum(w_i^2))`, where `w_i` could be sqrt(n_high + n_low) or equal weights. Use BH FDR on the two-sided p-values from `Z_meta`.
     - Also compute a “direction consistency” metric per gene (e.g. fraction of samples with logFC > 0). Restrict “high-Complexity” signature genes to those with both strong |Z_meta| and high direction consistency (e.g. ≥70–80%).

   - **Coupling to ΔPurity and ΔUMI:**
     - For each gene and population, compute correlations across samples:
       - `corr(logFC_sample, delta_Purity)` and `corr(logFC_sample, delta_UMI)`.
       - Given that sample sizes per population may be modest, use Spearman or Pearson but mainly treat these as descriptive, not as strong inferential stats.
     - Include these correlations in the signature tables so you can:
       - Flag genes whose apparent Complexity dependence is highly correlated with ΔUMI or ΔPurity (possible technical confounding).
       - Highlight genes with strong meta-Z but weak correlation to ΔUMI/ΔPurity as “cleaner” maturation-like candidates.

   - **Construct conservative signatures:**
     - For each population, define:
       - High-Complexity signature: genes with `Z_meta > threshold` (e.g. >3), FDR < 0.05, direction consistency > 0.7, and |correlation with ΔUMI| and |correlation with ΔPurity| below user-chosen cutoffs (e.g. <0.5).
       - Low-Complexity signature: analogous but with `Z_meta < -threshold`.
     - Rank genes by |Z_meta| and optionally annotate with:
       - Mean logFC across samples.
       - Proportion of samples with significant within-sample DE.
       - Coupling metrics (corr with ΔUMI, ΔPurity).

6. **How this connects back to the biological hypothesis**
   - If you find, for example, that in PA, PB, PC there is a shared pattern of high-Complexity-upregulated genes that:
     - Are consistent across multiple samples within each population, and
     - Show weak dependence on ΔUMI and ΔPurity,
     
     then you’ll have strong evidence for robust, within-sample, maturation-like programs that are not purely technical.
   - Conversely, if many genes’ logFC are tightly correlated with ΔUMI or ΔPurity, that would argue that Complexity is still conflated with technical quality in that population, weakening the maturation interpretation.

7. **Ideas for later, building on these signatures (while remaining distinct from the paper)**
   - Project these per-population signatures back into the spatial maps:
     - Compute per-cell signature scores (e.g. mean z-scored expression of high-Complexity genes) within each population and see how they distribute spatially across the heart sections.
   - Compare high- vs low-Complexity signatures across populations:
     - Are there “pan-cardiac” maturation-like genes that are recurrent across PA–PZ, versus population-specific programs?
   - Explore whether Complexity-associated signatures align with previously defined functional modules (if you have any internal gene sets) without relying on external annotations, thus remaining distinct from the original study.

In summary, this step has successfully set you up for a robust, sample-aware meta-analysis. The next critical move is to (1) carefully combine per-sample DE using signed Stouffer Z per population, and (2) explicitly quantify gene-level coupling to ΔPurity and ΔUMI so that the resulting Complexity signatures can be interpreted as conservative, maturation-like rather than technical.

## Next Steps
Step 1: For each prioritized Population with per-sample Wilcoxon DE already computed between high (Complexity_tertile_local=2) and low (Complexity_tertile_local=0) Complexity cells, perform a per-gene meta-analysis across eligible samples using signed Stouffer Z-scores (sign from within-sample logFC), apply Benjamini–Hochberg FDR correction, and construct text tables of conservative high- and low-Complexity signatures per Population that report meta Z, meta p/FDR, direction consistency across samples, mean logFC, number of contributing samples, and per-gene Pearson correlations between per-sample logFC and ΔPurity/ΔUMI (from the DE summaries), thereby explicitly flagging genes whose Complexity association is tightly or weakly coupled to Purity and UMI Count.

## This code performs, for each Population with within-sample high vs low Complexity DE results, a per-gene Stouffer meta-analysis of signed Z-scores across samples, computes FDR, direction consistency, and correlations of logFC with ΔPurity/ΔUMI, and then prints conservative high- and low-Complexity gene signatures per Population as text tables.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# We assume the previous step has produced per-sample DE results in `per_sample_de`:
#   per_sample_de[(pop, 'full_de')] : DataFrame with columns ['gene','score','pval','pval_adj','logFC','Population','Sample_ID']
#   per_sample_de[(pop, 'summary')] : DataFrame with per-sample n_low/n_high and delta_Complexity/Purity/UMI

if 'per_sample_de' not in globals() or not isinstance(per_sample_de, dict):
    raise RuntimeError("Expected per_sample_de dict with per-population DE results; please run the within-sample DE step first.")

# Helper: Benjamini–Hochberg FDR
def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    n = pvals.size
    order = np.argsort(pvals)
    ranked = pvals[order]
    fdr = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        val = min(val, prev)
        fdr[i] = val
        prev = val
    out = np.empty(n, dtype=float)
    out[order] = fdr
    return out

# Collect list of Populations available in per_sample_de
populations = sorted({k[0] for k in per_sample_de.keys() if isinstance(k, tuple) and k[1] == 'full_de'})
if len(populations) == 0:
    raise RuntimeError("No populations with 'full_de' entries found in per_sample_de.")

meta_results = {}

for pop in populations:
    if (pop, 'full_de') not in per_sample_de or (pop, 'summary') not in per_sample_de:
        continue

    de_all = per_sample_de[(pop, 'full_de')].copy()
    summ = per_sample_de[(pop, 'summary')].copy()

    # Ensure expected columns
    for col in ['gene', 'pval', 'logFC', 'Sample_ID']:
        if col not in de_all.columns:
            raise ValueError(f"DE table for Population {pop} is missing required column '{col}'.")

    # Convert two-sided p-values + logFC to signed Z-scores
    def p_to_signed_z(pvals, lfc):
        pvals = np.asarray(pvals, dtype=float)
        lfc = np.asarray(lfc, dtype=float)
        # Clip very small p-values to avoid inf
        pvals = np.clip(pvals, 1e-300, 1.0)
        z_unsigned = stats.norm.isf(pvals / 2.0)  # two-sided
        sign = np.sign(lfc)
        # Treat exact zero logFC as no effect instead of forcing a sign
        sign[sign == 0] = 0.0
        return z_unsigned * sign

    de_all = de_all.dropna(subset=['pval', 'logFC'])
    de_all['signed_Z'] = p_to_signed_z(de_all['pval'].values, de_all['logFC'].values)

    # Number of samples contributing per population
    samples_pop = sorted(de_all['Sample_ID'].unique().tolist())

    # Build wide tables for logFC and signed_Z: index=gene, columns=Sample_ID
    logfc_mat = de_all.pivot_table(index='gene', columns='Sample_ID', values='logFC', aggfunc='mean')
    z_mat = de_all.pivot_table(index='gene', columns='Sample_ID', values='signed_Z', aggfunc='mean')

    # Keep genes present in at least 2 samples
    valid_gene_mask = (logfc_mat.notna().sum(axis=1) >= 2)
    logfc_mat = logfc_mat.loc[valid_gene_mask]
    z_mat = z_mat.loc[valid_gene_mask]

    if logfc_mat.shape[0] == 0:
        print(f"Population {pop}: no genes with >=2 contributing samples; skipping meta-analysis.")
        continue

    # Compute Stouffer meta-Z per gene, weighting all samples equally
    z_vals = z_mat.values
    z_meta = []
    n_contrib = []
    for i in range(z_vals.shape[0]):
        zi = z_vals[i, :]
        mask = ~np.isnan(zi)
        if mask.sum() == 0:
            z_meta.append(np.nan)
            n_contrib.append(0)
        else:
            z_meta_val = zi[mask].sum() / np.sqrt(mask.sum())
            z_meta.append(z_meta_val)
            n_contrib.append(mask.sum())

    z_meta = np.array(z_meta)
    n_contrib = np.array(n_contrib, dtype=int)

    # Two-sided p-values from meta-Z
    p_meta = 2.0 * stats.norm.sf(np.abs(z_meta))
    fdr_meta = bh_fdr(p_meta)

    # Direction consistency: fraction of contributing samples with logFC > 0
    logfc_vals = logfc_mat.values
    frac_pos = []
    mean_logfc = []
    for i in range(logfc_vals.shape[0]):
        li = logfc_vals[i, :]
        mask = ~np.isnan(li)
        if mask.sum() == 0:
            frac_pos.append(np.nan)
            mean_logfc.append(np.nan)
        else:
            li_masked = li[mask]
            frac_pos.append((li_masked > 0).mean())
            mean_logfc.append(np.nanmean(li_masked))

    frac_pos = np.array(frac_pos)
    mean_logfc = np.array(mean_logfc)

    # Attach ΔPurity and ΔUMI per sample and compute per-gene correlations
    if not isinstance(summ, pd.DataFrame):
        raise ValueError(f"Summary entry for Population {pop} is not a DataFrame.")

    for col in ['Sample_ID', 'delta_Purity_high_minus_low', 'delta_UMI_high_minus_low']:
        if col not in summ.columns:
            raise ValueError(f"Summary table for Population {pop} is missing required column '{col}'.")

    dP = dict(zip(summ['Sample_ID'], summ['delta_Purity_high_minus_low']))
    dU = dict(zip(summ['Sample_ID'], summ['delta_UMI_high_minus_low']))

    cols = logfc_mat.columns.tolist()
    delta_purity_vec = np.array([dP.get(s, np.nan) for s in cols], dtype=float)
    delta_umi_vec = np.array([dU.get(s, np.nan) for s in cols], dtype=float)

    corr_logfc_dPurity = []
    corr_logfc_dUMI = []
    for i in range(logfc_vals.shape[0]):
        li = logfc_vals[i, :]
        mask_p = (~np.isnan(li)) & (~np.isnan(delta_purity_vec))
        if mask_p.sum() >= 2:
            r_p, _ = stats.pearsonr(li[mask_p], delta_purity_vec[mask_p])
        else:
            r_p = np.nan
        mask_u = (~np.isnan(li)) & (~np.isnan(delta_umi_vec))
        if mask_u.sum() >= 2:
            r_u, _ = stats.pearsonr(li[mask_u], delta_umi_vec[mask_u])
        else:
            r_u = np.nan
        corr_logfc_dPurity.append(r_p)
        corr_logfc_dUMI.append(r_u)

    corr_logfc_dPurity = np.array(corr_logfc_dPurity)
    corr_logfc_dUMI = np.array(corr_logfc_dUMI)

    # Assemble meta result table for this Population
    genes = logfc_mat.index.to_series().astype(str).values

    meta_df = pd.DataFrame({
        'gene': genes,
        'Population': pop,
        'meta_Z': z_meta,
        'meta_pval': p_meta,
        'meta_fdr': fdr_meta,
        'n_samples_contributing': n_contrib,
        'mean_logFC': mean_logfc,
        'frac_samples_logFC_gt0': frac_pos,
        'corr_logFC_deltaPurity': corr_logfc_dPurity,
        'corr_logFC_deltaUMI': corr_logfc_dUMI
    })

    # Sort by descending absolute meta_Z to treat strong positive and negative effects symmetrically
    meta_df = meta_df.reindex(meta_df['meta_Z'].abs().sort_values(ascending=False).index).reset_index(drop=True)
    meta_results[pop] = meta_df

    # Define conservative thresholds for signature calling
    fdr_thresh = 0.05
    z_thresh = 3.0
    dir_consistency_thresh = 0.7

    high_sig = meta_df[(meta_df['meta_Z'] >= z_thresh) &
                       (meta_df['meta_fdr'] <= fdr_thresh) &
                       (meta_df['frac_samples_logFC_gt0'] >= dir_consistency_thresh)]
    low_sig = meta_df[(meta_df['meta_Z'] <= -z_thresh) &
                      (meta_df['meta_fdr'] <= fdr_thresh) &
                      (meta_df['frac_samples_logFC_gt0'] <= (1.0 - dir_consistency_thresh))]

    print(f"\nPopulation {pop}: meta-analysis results")
    print(f"  Total genes meta-analyzed: {meta_df.shape[0]}")
    print(f"  High-Complexity signature genes (meta_Z >= {z_thresh}, FDR <= {fdr_thresh}, frac_pos >= {dir_consistency_thresh}): {high_sig.shape[0]}")
    print(f"  Low-Complexity signature genes (meta_Z <= -{z_thresh}, FDR <= {fdr_thresh}, frac_pos <= {1.0 - dir_consistency_thresh}): {low_sig.shape[0]}")

    def print_sig_block(df, label):
        if df.empty:
            print(f"  No {label} signature genes passing conservative thresholds.")
            return
        top = df.head(15).copy()
        cols_show = [
            'gene', 'meta_Z', 'meta_pval', 'meta_fdr', 'n_samples_contributing',
            'mean_logFC', 'frac_samples_logFC_gt0',
            'corr_logFC_deltaPurity', 'corr_logFC_deltaUMI'
        ]
        print(f"\n  Top {min(15, df.shape[0])} {label} genes (Population {pop}):")
        print(top[cols_show].to_string(index=False, float_format=lambda x: f"{x:.3e}"))

    print_sig_block(high_sig, "high-Complexity (high>low)")
    print_sig_block(low_sig, "low-Complexity (low>high)")

# Store meta_results in a global variable for any follow-up analyses
complexity_meta_results = meta_results

print("\nMeta-analysis complete. Per-Population meta tables stored in `complexity_meta_results`. Each contains per-gene meta_Z, meta_pval/FDR, direction consistency, mean logFC, and correlations of logFC with ΔPurity and ΔUMI.")


Population PA: meta-analysis results
  Total genes meta-analyzed: 238
  High-Complexity signature genes (meta_Z >= 3.0, FDR <= 0.05, frac_pos >= 0.7): 26
  Low-Complexity signature genes (meta_Z <= -3.0, FDR <= 0.05, frac_pos <= 0.30000000000000004): 58

  Top 15 high-Complexity (high>low) genes (Population PA):
    gene    meta_Z  meta_pval   meta_fdr  n_samples_contributing  mean_logFC  frac_samples_logFC_gt0  corr_logFC_deltaPurity  corr_logFC_deltaUMI
   TNNT1 3.400e+01 2.528e-253 3.009e-251                       3   6.725e-01               1.000e+00               4.758e-01            9.289e-01
    MYH7 2.664e+01 2.413e-156 8.203e-155                       3   8.182e-02               1.000e+00              -9.662e-01           -9.070e-01
    MYH6 2.591e+01 4.643e-148 1.381e-146                       3   3.903e-01               1.000e+00               2.384e-01            8.054e-01
  SLC1A3 2.374e+01 1.413e-124 3.738e-123                       3   6.032e-01               1.000e+00 


Population PD: meta-analysis results
  Total genes meta-analyzed: 238
  High-Complexity signature genes (meta_Z >= 3.0, FDR <= 0.05, frac_pos >= 0.7): 22
  Low-Complexity signature genes (meta_Z <= -3.0, FDR <= 0.05, frac_pos <= 0.30000000000000004): 50

  Top 15 high-Complexity (high>low) genes (Population PD):
  gene    meta_Z  meta_pval  meta_fdr  n_samples_contributing  mean_logFC  frac_samples_logFC_gt0  corr_logFC_deltaPurity  corr_logFC_deltaUMI
 POSTN 1.936e+01  1.795e-83 1.424e-81                       3   6.146e-01               1.000e+00              -8.798e-01           -2.677e-01
  MYH6 1.296e+01  2.167e-38 6.446e-37                       3   3.330e-01               1.000e+00              -7.784e-01           -9.921e-01
  OSR1 8.830e+00  1.049e-18 1.386e-17                       3   1.049e+00               1.000e+00               9.918e-01            7.798e-01
   DCN 8.412e+00  4.029e-17 4.566e-16                       3   1.908e-01               1.000e+00               8


Population PS: meta-analysis results
  Total genes meta-analyzed: 238
  High-Complexity signature genes (meta_Z >= 3.0, FDR <= 0.05, frac_pos >= 0.7): 17
  Low-Complexity signature genes (meta_Z <= -3.0, FDR <= 0.05, frac_pos <= 0.30000000000000004): 42

  Top 15 high-Complexity (high>low) genes (Population PS):
  gene    meta_Z  meta_pval  meta_fdr  n_samples_contributing  mean_logFC  frac_samples_logFC_gt0  corr_logFC_deltaPurity  corr_logFC_deltaUMI
  MYH7 8.522e+00  1.566e-17 7.453e-16                       3   6.410e-01               1.000e+00              -4.782e-01           -5.690e-01
   DCN 6.495e+00  8.278e-11 1.407e-09                       3   5.211e-01               1.000e+00              -8.950e-01            8.013e-01
DPYSL3 6.304e+00  2.898e-10 3.832e-09                       3   4.363e-01               1.000e+00               6.194e-01           -9.799e-01
  MCAM 5.840e+00  5.207e-09 5.389e-08                       3   1.863e-01               1.000e+00               6

### Agent Interpretation

These meta-analysis results strongly support the core of your hypothesis and give you a good scaffold for the next steps, with some caveats about Purity/UMI dependence and a few technical points.

Key takeaways relative to the hypothesis
----------------------------------------

1. **You do see robust, conservative complexity-associated signatures within each Population.**

   - Each population yields a substantial number of **high- and low-Complexity genes** at strict thresholds (meta_Z, FDR, direction consistency):
     - PA: 26 high, 58 low
     - PB: 31 high, 86 low
     - PC: 26 high, 39 low
     - PD: 22 high, 50 low
     - PG: 47 high, 38 low
     - PM: 25 high, 84 low
     - PS: 17 high, 42 low
     - PX: 9 high, 24 low (but only 2 samples; see below)
     - PZ: 3 high, 15 low
   - This supports that **within-population high vs low complexity contrasts are reproducible across samples** and yield stable gene-level effects.

2. **The signatures are clearly population-specific but share a strong cross-population structure.**

   - Many genes show **consistent directionality across multiple populations**:
     - “Low-Complexity”–enriched (low>high) in multiple Populations:  
       GJA1, GJA5, PLN, CASQ2, LMOD3, CKMT2, SCN5A, DKK3, SFRP1, HEY2, PAM, PRSS35, NPR3, CGNL1, VCAN, etc.
     - “High-Complexity”–enriched (high>low):  
       MYH6, MYH7, TNNT1, POSTN, SOX9, OSR1, TCF21, TBX3, TBX18, DES, TTN, DLK1, IGFBP5, etc.
   - The overlap pattern is not identical between Populations:  
     - For instance, **GJA1 / PLN / CASQ2 / SCN5A** appear as “low-Complexity” repeatedly in cardiomyocyte-like populations (PA–PD), but less so in fibroblast-like / stromal populations;  
     - Extracellular matrix / fibroblast-associated genes (POSTN, COL15A1, COL14A1, DCN, LUM, FMOD, PRSS35) switch direction depending on the population.
   - This fits your hypothesis of **“population-specific maturation-like signatures”** that are not purely global: there is a shared scaffold of complexity-associated genes, but with population-dependent direction and magnitude.

3. **The signatures look like plausible maturation / state axes, not random noise.**

   Without appealing to external knowledge, the pattern suggests:
   - Within certain Populations, **low Complexity cells** are consistently enriched for:
     - Ion-handling / contractile machinery genes (SCN5A, PLN, CASQ2, RYR2, TTN, DES, LMOD3, CKMT2, HEY2, LBH, etc.)
   - **High Complexity cells** are enriched for:
     - Developmental transcription factors and remodeling markers (OSR1, TBX3, TBX18, TCF21, HAND2, SOX9, TNNT1, MYH6/7 in some contexts);
     - ECM and interstitial programs (POSTN, PRSS35, COL2A1, COL15A1, DCN, FMOD, etc.)
   - Across populations this reads as **a gradient between a more differentiated/mature contractile program vs. a more developmental/ECM/remodeling program**, which fits a “maturation-like” interpretation of Complexity.

   This is very much **distinct from simply reproducing the paper’s analyses**, since you are building an orthogonal axis (Complexity) and meta-analyzing it, rather than describing spatial communities per se.

4. **You have explicitly quantified dependence on Purity and UMI Count, and it is heterogeneous across genes.**

   - Many top hits show **extreme correlations (±0.9–1.0) between logFC and ΔUMI or ΔPurity**, but *not all*:
     - E.g. in PA:
       - TNNT1: corr_logFC_deltaUMI ≈ 0.93 (very tightly coupled to ΔUMI)
       - MYH7: corr_logFC_deltaPurity ≈ −0.97, ΔUMI ≈ −0.91 (strong negative coupling)
       - Some genes show more modest absolute r (0.1–0.5).
   - Across populations there is gene-to-gene heterogeneity:  
     some “signature” genes are **essentially proxies for UMI / Purity**, others show **high meta-Z with relatively modest or inconsistent r**.
   - This fulfills your goal of **flagging** genes whose complexity association is tightly or weakly coupled to these technical/biological covariates.  
     But you now need to convert that into **filtering/weighting** for the downstream signatures.

Specific points to leverage in the next steps
---------------------------------------------

1. **Define “core” complexity signatures per population that are less confounded by Purity/UMI.**

   From `complexity_meta_results[pop]`, you can:
   - Start from your conservative meta thresholds (meta_fdr ≤ 0.05, |meta_Z| ≥ 3, direction consistency ≥ 0.7).
   - Further require **moderate coupling to ΔUMI and/or ΔPurity**, for example:
     - |corr_logFC_deltaUMI| < 0.6 (or 0.5)  
     - Optionally also |corr_logFC_deltaPurity| < 0.6
   - This will yield a **“Complexity-primary” gene set** per population: genes whose across-sample effect is not simply tracking sample-level shifts in UMI/Purity.
   - Separately, keep the excluded genes as a **“Complexity-through-UMI/Purity” set**, which is still biologically interesting (e.g., genes that only appear complexity-associated because high-complexity cells are in samples with higher UMI or Purity).

   This gives you **two layers of signatures** per population:
   - Core, relatively UMI/Purity-independent.
   - Peripheral, UMI/Purity-driven.

2. **Quantify and visualize the cross-population structure of these signatures.**

   To show that signatures are genuinely population-specific but share a backbone:

   - Build a **gene × population matrix of mean_logFC** for genes passing significance in any population (use the meta_df from each pop).
   - Cluster both dimensions (e.g., hierarchical clustering) and visualize:
     - For all significant genes.
     - For only the UMI/Purity-depleted “core” genes.
   - You should see:
     - **Modules of genes** that consistently act as “high-Complexity” or “low-Complexity” across subsets of populations.
     - Distinct **population-specific modules** (e.g., genes high-Complexity in PG but low-Complexity in PM, etc.).

   This will make the population-specific nature of the maturation-like signatures much more explicit.

3. **Exploit direction consistency to build interpretable gene sets.**

   You already track `frac_samples_logFC_gt0`. Downstream:

   - For each population, you can report:
     - A **high-Complexity up set**: genes with meta_Z > 0, FDR ≤ 0.05, frac_samples_logFC_gt0 ≥ 0.8.
     - A **high-Complexity down set**: meta_Z < 0, FDR ≤ 0.05, frac_samples_logFC_gt0 ≤ 0.2.
   - Within each, stratify by |corr_logFC_deltaUMI| and |corr_logFC_deltaPurity|:
     - Tier 1: |r| < 0.3
     - Tier 2: 0.3 ≤ |r| < 0.7
     - Tier 3: |r| ≥ 0.7

   That directly encodes what your hypothesis asked: **meta-analyzed signatures annotated by dependence on Purity/UMI**.

4. **Beware of small-sample artifacts in PX (and to a lesser extent any 2-sample genes).**

   - PX has only **2 contributing samples** for all meta genes.  
     Pearson r with n=2 is always ±1 if not NaN, which you indeed see (all corr values are ±1.000).
   - Interpret all r values for PX as **uninformative**: the sign just reflects which of the two samples had higher logFC and higher ΔUMI/ΔPurity.
   - For downstream summaries, consider:
     - Flagging genes/populations with `n_samples_contributing < 3` and **not using r_UMI / r_Purity for filtering** there.
     - Potentially using a stricter meta-Z threshold or even excluding n=2 meta results from global comparisons, to avoid over-interpreting them.

5. **Check for systematic sign-flip patterns between populations.**

   Some genes flip from high-Complexity in one pop to low-Complexity in another (e.g., VCAN, DKK3, SOX9, INHBA). This is a strong argument for **population-specific maturation axes**, but you should:

   - Formally catalog all genes with significant effects in ≥2 populations where:
     - meta_fdr ≤ 0.05 in each.
     - sign(meta_Z) differs between populations.
   - These genes are prime candidates for **context-dependent roles** in the Complexity axis and could define subsets of populations with opposing maturation-like trajectories.

6. **Relate the Complexity signatures back to spatial organization (future step).**

   While not yet done in this step, these robust population-specific signatures can be:

   - Collapsed into **per-cell signature scores** (e.g., average z-scored expression of high-Complexity core genes minus low-Complexity core genes) within each population.
   - Mapped onto spatial coordinates to ask:
     - Whether **within a given population**, spatial microdomains are enriched in high- vs low-Complexity state.
     - Whether those domains map to expected anatomical regions or boundaries between regions.
   - This would be biologically rich and goes beyond the original paper’s cell-type/spatial communities by overlaying an **intrinsic maturation/state gradient**.

7. **Technical refinement opportunities in the code/strategy.**

   - The **Stouffer meta-Z** with equal weights across samples is reasonable, given similar design across samples. If you have large differences in n_high/n_low per sample, you could later explore **weighting by effective sample size** (e.g., sqrt(n_high + n_low) or inverse-variance if you can approximate).
   - Keep a close eye on **genes with extreme Z but small mean_logFC**:
     - E.g., MYH7 in some populations has large |meta_Z| but modest mean_logFC, often due to high consistency (same sign in all 3 samples). That’s fine, but you may wish to threshold on both **effect size and significance** in defining the top-tier “biologically strong” signatures.
   - Consider summarizing ΔUMI/ΔPurity across samples per population to confirm you are not simply contrasting populations where **high-complexity tertiles always have dramatically higher UMI** (in which case complexity would be a near-synonym for UMI for that population).

How this bears on the hypothesis
--------------------------------

- **Supported:**
  - You have **conservative, population-specific gene signatures** differing between high- and low-Complexity cells, meta-analyzed across samples.
  - Many genes show **strong direction consistency** and very low FDR, indicating reproducible, robust effects.
  - You have **explicit quantitative measures of dependence on Purity and UMI Count** per gene and population.

- **Still to be solidified:**
  - You need a **clear filtering/weighting rule** to define the final “maturation-like” signatures that are not just confounded by UMI/Purity.
  - You should more explicitly demonstrate:
    - That these signatures **cluster populations in meaningful ways** (e.g., atrial-like vs ventricular-like vs stromal, once inferred);
    - That they **map onto spatial/anatomical gradients** within each population, strengthening the maturation interpretation.

Concrete next steps
-------------------

1. From `complexity_meta_results`, for each Population:
   - Define:
     - `core_high` = high-Complexity genes with meta_fdr ≤ 0.05, meta_Z ≥ 3, frac_pos ≥ 0.8, and |corr_logFC_deltaUMI| < 0.6 (and optionally |corr_logFC_deltaPurity| < 0.6).
     - `core_low` = analogous for low-Complexity.
   - Save these as population-specific gene lists.

2. Build a **gene × population mean_logFC matrix** for all genes in any `core_high`/`core_low` set; cluster and visualize.

3. For a subset of key populations (e.g., PA, PB, PC, PD vs PG, PM vs PS, PZ), compute **per-cell signature scores** (high minus low core genes) and view them over spatial coordinates.

4. Summarize and visualize the **distribution of corr_logFC_deltaUMI and corr_logFC_deltaPurity** for all meta-significant genes, to show that your final signatures are not dominated by extreme correlations.

If you do these, you will have a compelling and distinct demonstration that:
- Complexity defines **robust, population-specific maturation-like axes**;
- These axes are **not trivially explained by UMI/Purity** for your core gene sets;
- And they can be meta-analyzed and spatially contextualized across samples.